In [2]:
# -------------------------------------------------------------------------
# STANDARD LIBRARY
# -------------------------------------------------------------------------

from pathlib import Path
from importlib.metadata import version
import inspect
import json
import sys
import time
import warnings

# -------------------------------------------------------------------------
# CORE LIBRARIES
# -------------------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------------------------------------------------------
# SCIKIT-LEARN
# -------------------------------------------------------------------------

import sklearn

from sklearn.base import clone

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.model_selection import (
    TimeSeriesSplit,
)

from sklearn.tree import (
    DecisionTreeClassifier,
    DecisionTreeRegressor,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
)

# -------------------------------------------------------------------------
# FIGS
# -------------------------------------------------------------------------

from imodels import (
    FIGSClassifier,
    FIGSRegressor,
)

# -------------------------------------------------------------------------
# DISPLAY / REPRODUCIBILITY
# -------------------------------------------------------------------------

pd.set_option(
    "display.max_columns",
    250,
)

pd.set_option(
    "display.max_rows",
    150,
)

pd.set_option(
    "display.width",
    220,
)

pd.set_option(
    "display.float_format",
    lambda value: f"{value:,.4f}",
)

RANDOM_STATE = 42

np.random.seed(
    RANDOM_STATE
)

# -------------------------------------------------------------------------
# ENVIRONMENT REPORT
# -------------------------------------------------------------------------

print("Notebook 07 environment configured.")
print(f"Python        : {sys.executable}")
print(f"NumPy         : {np.__version__}")
print(f"pandas        : {pd.__version__}")
print(f"scikit-learn : {sklearn.__version__}")
print(f"imodels      : {version('imodels')}")

Notebook 07 environment configured.
Python        : /home/sina/anaconda3/envs/figs07/bin/python
NumPy         : 2.4.6
pandas        : 3.0.5
scikit-learn : 1.9.0
imodels      : 2.0.4


In [3]:
# -------------------------------------------------------------------------
# INSTALLED FIGS API INSPECTION
# -------------------------------------------------------------------------

figs_classifier_signature = inspect.signature(
    FIGSClassifier
)

figs_regressor_signature = inspect.signature(
    FIGSRegressor
)

print("FIGSClassifier:")
print(
    figs_classifier_signature
)

print("\nFIGSRegressor:")
print(
    figs_regressor_signature
)

FIGSClassifier:
(max_rules: int = 12, max_trees: int = None, min_impurity_decrease: float = 0.0, random_state=None, max_features: str = None, max_depth: int = None)

FIGSRegressor:
(max_rules: int = 12, max_trees: int = None, min_impurity_decrease: float = 0.0, random_state=None, max_features: str = None, max_depth: int = None)


In [4]:
# -------------------------------------------------------------------------
# NOTEBOOK 05 INPUTS
# -------------------------------------------------------------------------

MODELING_DATASET_PATH = Path(
    "../data/processed/"
    "prematch_modeling_dataset_laliga_2015_2016.parquet"
)

FEATURE_MANIFEST_PATH = Path(
    "../data/processed/"
    "prematch_modeling_feature_manifest_laliga_2015_2016.csv"
)

# -------------------------------------------------------------------------
# NOTEBOOK 06 INPUTS
# -------------------------------------------------------------------------

BASELINE_RESULTS_DIR = Path(
    "../data/processed/baseline_results"
)

EXPERIMENT_CONFIG_PATH = (
    BASELINE_RESULTS_DIR
    / "prematch_baseline_experiment_config.json"
)

BASELINE_SELECTION_PATH = (
    BASELINE_RESULTS_DIR
    / "prematch_baseline_model_selection.json"
)

CLASSIFICATION_BASELINE_RESULTS_PATH = (
    BASELINE_RESULTS_DIR
    / "prematch_classification_calibration_results.csv"
)

REGRESSION_BASELINE_RESULTS_PATH = (
    BASELINE_RESULTS_DIR
    / "prematch_regression_calibration_results.csv"
)

# -------------------------------------------------------------------------
# NOTEBOOK 07 OUTPUTS
# -------------------------------------------------------------------------

FIGS_RESULTS_DIR = Path(
    "../data/processed/figs_results"
)

FIGS_MODEL_DIR = Path(
    "../models/figs"
)

FIGS_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FIGS_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# -------------------------------------------------------------------------
# INPUT VALIDATION
# -------------------------------------------------------------------------

for required_path in [
    MODELING_DATASET_PATH,
    FEATURE_MANIFEST_PATH,
    EXPERIMENT_CONFIG_PATH,
    BASELINE_SELECTION_PATH,
    CLASSIFICATION_BASELINE_RESULTS_PATH,
    REGRESSION_BASELINE_RESULTS_PATH,
]:

    if not required_path.exists():

        raise FileNotFoundError(
            "Required upstream artifact not found:\n"
            f"{required_path.resolve()}"
        )

print("Notebook 07 paths configured.")

Notebook 07 paths configured.


In [5]:
# -------------------------------------------------------------------------
# MODELING DATA
# -------------------------------------------------------------------------

modeling_df = pd.read_parquet(
    MODELING_DATASET_PATH
)

feature_manifest_df = pd.read_csv(
    FEATURE_MANIFEST_PATH
)

modeling_df[
    "clean_date"
] = pd.to_datetime(
    modeling_df[
        "clean_date"
    ],
    errors="raise",
)

modeling_df = (
    modeling_df
    .sort_values(
        [
            "clean_date",
            "match_id",
        ]
    )
    .reset_index(drop=True)
)

# -------------------------------------------------------------------------
# FROZEN NOTEBOOK 06 CONFIGURATION
# -------------------------------------------------------------------------

with open(
    EXPERIMENT_CONFIG_PATH,
    "r",
    encoding="utf-8",
) as file_handle:

    experiment_config = json.load(
        file_handle
    )

with open(
    BASELINE_SELECTION_PATH,
    "r",
    encoding="utf-8",
) as file_handle:

    baseline_selection = json.load(
        file_handle
    )

classification_baseline_results_df = (
    pd.read_csv(
        CLASSIFICATION_BASELINE_RESULTS_PATH
    )
)

regression_baseline_results_df = (
    pd.read_csv(
        REGRESSION_BASELINE_RESULTS_PATH
    )
)

# -------------------------------------------------------------------------
# TEST MUST STILL BE SEALED
# -------------------------------------------------------------------------

assert (
    experiment_config[
        "test_set_opened"
    ]
    is False
)

assert (
    baseline_selection[
        "test_set_opened"
    ]
    is False
)

print("Notebook 06 frozen protocol loaded.")

print(
    "Baseline classifier:",
    baseline_selection[
        "classification"
    ][
        "selected_model"
    ],
)

print(
    "Baseline regressor :",
    baseline_selection[
        "regression"
    ][
        "selected_model"
    ],
)

print(
    "Held-out test      : SEALED"
)

Notebook 06 frozen protocol loaded.
Baseline classifier: RandomForest
Baseline regressor : RandomForest
Held-out test      : SEALED


In [6]:
# -------------------------------------------------------------------------
# NORMALIZE PREDICTOR FLAG
# -------------------------------------------------------------------------

predictor_flag = (
    feature_manifest_df[
        "used_as_predictor"
    ]
)

if predictor_flag.dtype != bool:

    predictor_flag = (
        predictor_flag
        .astype(str)
        .str.lower()
        .eq("true")
    )

# -------------------------------------------------------------------------
# CONVENTIONAL FEATURES
# -------------------------------------------------------------------------

CONVENTIONAL_FEATURE_COLUMNS = (
    feature_manifest_df.loc[
        predictor_flag
        & feature_manifest_df[
            "feature_family"
        ]
        .astype(str)
        .str.lower()
        .eq("conventional"),
        "column",
    ]
    .tolist()
)

# -------------------------------------------------------------------------
# NETWORK FEATURES
# -------------------------------------------------------------------------

NETWORK_FEATURE_COLUMNS = (
    feature_manifest_df.loc[
        predictor_flag
        & feature_manifest_df[
            "feature_family"
        ]
        .astype(str)
        .str.lower()
        .eq("network"),
        "column",
    ]
    .tolist()
)

FULL_FEATURE_COLUMNS = (
    CONVENTIONAL_FEATURE_COLUMNS
    + NETWORK_FEATURE_COLUMNS
)

assert len(
    CONVENTIONAL_FEATURE_COLUMNS
) == 6

assert len(
    NETWORK_FEATURE_COLUMNS
) == 180

assert len(
    FULL_FEATURE_COLUMNS
) == 186

print("Feature families recovered.")

print(
    f"Conventional : "
    f"{len(CONVENTIONAL_FEATURE_COLUMNS)}"
)

print(
    f"Network      : "
    f"{len(NETWORK_FEATURE_COLUMNS)}"
)

print(
    f"Combined     : "
    f"{len(FULL_FEATURE_COLUMNS)}"
)

Feature families recovered.
Conventional : 6
Network      : 180
Combined     : 186


In [7]:
# -------------------------------------------------------------------------
# FROZEN BOUNDARIES
# -------------------------------------------------------------------------

TEST_START_DATE = pd.Timestamp(
    experiment_config[
        "test_start_date"
    ]
)

CALIBRATION_START_DATE = pd.Timestamp(
    experiment_config[
        "calibration_start_date"
    ]
)

# -------------------------------------------------------------------------
# PARTITIONS
# -------------------------------------------------------------------------

training_df = (
    modeling_df.loc[
        modeling_df[
            "clean_date"
        ] < CALIBRATION_START_DATE
    ]
    .copy()
    .reset_index(drop=True)
)

validation_df = (
    modeling_df.loc[
        (
            modeling_df[
                "clean_date"
            ] >= CALIBRATION_START_DATE
        )
        & (
            modeling_df[
                "clean_date"
            ] < TEST_START_DATE
        )
    ]
    .copy()
    .reset_index(drop=True)
)

sealed_test_df = (
    modeling_df.loc[
        modeling_df[
            "clean_date"
        ] >= TEST_START_DATE
    ]
    .copy()
    .reset_index(drop=True)
)

# -------------------------------------------------------------------------
# VALIDATE AGAINST NOTEBOOK 06
# -------------------------------------------------------------------------

assert len(
    training_df
) == baseline_selection[
    "development_protocol"
][
    "model_selection_matches"
]

assert len(
    validation_df
) == baseline_selection[
    "development_protocol"
][
    "validation_matches"
]

assert len(
    sealed_test_df
) == baseline_selection[
    "development_protocol"
][
    "sealed_test_matches"
]

assert (
    training_df[
        "clean_date"
    ].max()
    <
    validation_df[
        "clean_date"
    ].min()
)

assert (
    validation_df[
        "clean_date"
    ].max()
    <
    sealed_test_df[
        "clean_date"
    ].min()
)

print("Frozen chronological protocol reconstructed.")

print(
    f"Training    : {len(training_df):,}"
)

print(
    f"Validation  : {len(validation_df):,}"
)

print(
    f"SEALED test : {len(sealed_test_df):,}"
)

Frozen chronological protocol reconstructed.
Training    : 240
Validation  : 61
SEALED test : 79


In [8]:
# -------------------------------------------------------------------------
# CLASS ENCODING
# -------------------------------------------------------------------------

CLASS_LABELS = [
    "A",
    "D",
    "H",
]

CLASS_TO_INT = {
    "A": 0,
    "D": 1,
    "H": 2,
}

for dataframe in [
    training_df,
    validation_df,
    sealed_test_df,
]:

    dataframe[
        "target_outcome_int"
    ] = (
        dataframe[
            "target_outcome"
        ]
        .map(
            CLASS_TO_INT
        )
        .astype(
            "int8"
        )
    )

# -------------------------------------------------------------------------
# FULL DEVELOPMENT FEATURE MATRICES
# -------------------------------------------------------------------------

X_train_full = (
    training_df[
        FULL_FEATURE_COLUMNS
    ]
    .copy()
)

X_validation_full = (
    validation_df[
        FULL_FEATURE_COLUMNS
    ]
    .copy()
)

# -------------------------------------------------------------------------
# TARGETS
# -------------------------------------------------------------------------

y_class_train = (
    training_df[
        "target_outcome_int"
    ]
    .to_numpy()
)

y_class_validation = (
    validation_df[
        "target_outcome_int"
    ]
    .to_numpy()
)

y_margin_train = (
    training_df[
        "target_margin"
    ]
    .to_numpy(
        dtype=float
    )
)

y_margin_validation = (
    validation_df[
        "target_margin"
    ]
    .to_numpy(
        dtype=float
    )
)

# -------------------------------------------------------------------------
# TEST REMAINS SEALED
# -------------------------------------------------------------------------

assert "X_test" not in globals()
assert "y_class_test" not in globals()
assert "y_margin_test" not in globals()

print("Development matrices created.")

print(
    "X_train_full:",
    X_train_full.shape
)

print(
    "X_validation_full:",
    X_validation_full.shape
)

print(
    "Final test matrices remain sealed."
)

Development matrices created.
X_train_full: (240, 186)
X_validation_full: (61, 186)
Final test matrices remain sealed.


In [9]:
# -------------------------------------------------------------------------
# SMOKE-TEST IMPUTATION
# -------------------------------------------------------------------------

smoke_imputer = SimpleImputer(
    strategy="median",
    keep_empty_features=True,
)

X_smoke_train = (
    smoke_imputer.fit_transform(
        X_train_full
    )
)

X_smoke_validation = (
    smoke_imputer.transform(
        X_validation_full
    )
)

assert X_smoke_train.shape == (
    len(training_df),
    186,
)

assert X_smoke_validation.shape == (
    len(validation_df),
    186,
)

assert np.isfinite(
    X_smoke_train
).all()

assert np.isfinite(
    X_smoke_validation
).all()

print("Smoke-test imputation passed.")

Smoke-test imputation passed.


In [10]:
# -------------------------------------------------------------------------
# FIGS REGRESSOR SMOKE TEST
# -------------------------------------------------------------------------

figs_regressor_smoke = FIGSRegressor(
    max_rules=4,
    random_state=RANDOM_STATE,
)

figs_regressor_smoke.fit(
    X_smoke_train,
    y_margin_train,
)

smoke_margin_predictions = (
    figs_regressor_smoke.predict(
        X_smoke_validation[
            :10
        ]
    )
)

assert len(
    smoke_margin_predictions
) == 10

assert np.isfinite(
    smoke_margin_predictions
).all()

print(
    "FIGSRegressor smoke test passed."
)

print(
    "Sample predictions:"
)

print(
    smoke_margin_predictions
)

FIGSRegressor smoke test passed.
Sample predictions:
[ 0.17514124 -2.25        0.17514124  1.57142857  0.17514124  0.17514124
  0.17514124  0.17514124  0.17514124  0.17514124]


In [11]:
# -------------------------------------------------------------------------
# FIGS CLASSIFIER MULTICLASS SMOKE TEST
# -------------------------------------------------------------------------

figs_classifier_smoke = FIGSClassifier(
    max_rules=4,
    random_state=RANDOM_STATE,
)

native_multiclass_ok = True

multiclass_failure_reason = None

try:

    figs_classifier_smoke.fit(
        X_smoke_train,
        y_class_train,
    )

    learned_classes = np.asarray(
        figs_classifier_smoke.classes_
    )

    smoke_probabilities = (
        figs_classifier_smoke.predict_proba(
            X_smoke_validation[
                :20
            ]
        )
    )

    smoke_predictions = (
        figs_classifier_smoke.predict(
            X_smoke_validation[
                :20
            ]
        )
    )

    # ---------------------------------------------------------------------
    # VALIDATION
    # ---------------------------------------------------------------------

    if learned_classes.shape != (
        3,
    ):

        native_multiclass_ok = False

        multiclass_failure_reason = (
            "FIGSClassifier did not retain "
            "three classes."
        )

    elif smoke_probabilities.shape != (
        20,
        3,
    ):

        native_multiclass_ok = False

        multiclass_failure_reason = (
            "predict_proba did not return "
            "an (n, 3) matrix."
        )

    elif not np.all(
        np.isfinite(
            smoke_probabilities
        )
    ):

        native_multiclass_ok = False

        multiclass_failure_reason = (
            "Non-finite probabilities returned."
        )

    elif not np.allclose(
        smoke_probabilities.sum(
            axis=1
        ),
        1.0,
        atol=1e-6,
    ):

        native_multiclass_ok = False

        multiclass_failure_reason = (
            "Probability rows do not sum to one."
        )

    elif not set(
        np.unique(
            smoke_predictions
        )
    ).issubset(
        {
            0,
            1,
            2,
        }
    ):

        native_multiclass_ok = False

        multiclass_failure_reason = (
            "Invalid class labels returned."
        )

except Exception as exc:

    native_multiclass_ok = False

    multiclass_failure_reason = (
        f"{type(exc).__name__}: {exc}"
    )

print(
    "Native multiclass FIGS supported:",
    native_multiclass_ok,
)

if native_multiclass_ok:

    print(
        "Classes:",
        learned_classes,
    )

    print(
        "Probability shape:",
        smoke_probabilities.shape,
    )

    print(
        "First five row sums:",
        smoke_probabilities.sum(
            axis=1
        )[
            :5
        ],
    )

else:

    print(
        "Reason:",
        multiclass_failure_reason,
    )

Native multiclass FIGS supported: True
Classes: [0 1 2]
Probability shape: (20, 3)
First five row sums: [1. 1. 1. 1. 1.]


In [12]:
# -------------------------------------------------------------------------
# CLASSIFICATION METRICS
# -------------------------------------------------------------------------

N_CLASSES = 3


def align_class_probabilities(
    probabilities,
    learned_classes,
):
    """
    Align probability columns to the fixed class order:
        0 = Away
        1 = Draw
        2 = Home
    """

    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    learned_classes = np.asarray(
        learned_classes,
        dtype=int,
    )

    aligned = np.zeros(
        (
            probabilities.shape[0],
            N_CLASSES,
        ),
        dtype=float,
    )

    for source_column, class_value in enumerate(
        learned_classes
    ):

        aligned[
            :,
            class_value
        ] = probabilities[
            :,
            source_column
        ]

    return aligned


def ranked_probability_score(
    y_true,
    probabilities,
):
    """
    Multiclass Ranked Probability Score
    for ordered classes A / D / H.
    """

    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    one_hot = np.eye(
        N_CLASSES
    )[
        y_true
    ]

    cumulative_probabilities = (
        np.cumsum(
            probabilities,
            axis=1,
        )
    )

    cumulative_outcomes = (
        np.cumsum(
            one_hot,
            axis=1,
        )
    )

    squared_differences = (
        (
            cumulative_probabilities[
                :,
                :-1
            ]
            -
            cumulative_outcomes[
                :,
                :-1
            ]
        )
        ** 2
    )

    return float(
        np.mean(
            np.sum(
                squared_differences,
                axis=1,
            )
            / (
                N_CLASSES - 1
            )
        )
    )


def multiclass_brier_score(
    y_true,
    probabilities,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    one_hot = np.eye(
        N_CLASSES
    )[
        y_true
    ]

    return float(
        np.mean(
            np.sum(
                (
                    probabilities
                    - one_hot
                )
                ** 2,
                axis=1,
            )
        )
    )


def expected_calibration_error(
    y_true,
    probabilities,
    n_bins=10,
):
    """
    Top-label ECE.
    """

    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    predicted_class = np.argmax(
        probabilities,
        axis=1,
    )

    confidence = np.max(
        probabilities,
        axis=1,
    )

    correct = (
        predicted_class
        == y_true
    ).astype(float)

    bin_edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1,
    )

    ece = 0.0

    for bin_index in range(
        n_bins
    ):

        lower = bin_edges[
            bin_index
        ]

        upper = bin_edges[
            bin_index + 1
        ]

        if bin_index == (
            n_bins - 1
        ):

            in_bin = (
                (confidence >= lower)
                & (confidence <= upper)
            )

        else:

            in_bin = (
                (confidence >= lower)
                & (confidence < upper)
            )

        if not np.any(
            in_bin
        ):
            continue

        bin_accuracy = float(
            correct[
                in_bin
            ].mean()
        )

        bin_confidence = float(
            confidence[
                in_bin
            ].mean()
        )

        bin_weight = float(
            in_bin.mean()
        )

        ece += (
            bin_weight
            * abs(
                bin_accuracy
                - bin_confidence
            )
        )

    return float(
        ece
    )


def evaluate_classification_probabilities(
    y_true,
    probabilities,
):

    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    predictions = np.argmax(
        probabilities,
        axis=1,
    )

    return {
        "rps": ranked_probability_score(
            y_true,
            probabilities,
        ),
        "log_loss": log_loss(
            y_true,
            probabilities,
            labels=[
                0,
                1,
                2,
            ],
        ),
        "brier": multiclass_brier_score(
            y_true,
            probabilities,
        ),
        "ece": expected_calibration_error(
            y_true,
            probabilities,
        ),
        "accuracy": accuracy_score(
            y_true,
            predictions,
        ),
        "balanced_accuracy": (
            balanced_accuracy_score(
                y_true,
                predictions,
            )
        ),
        "macro_f1": f1_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0,
        ),
    }


print(
    "Classification metrics configured."
)

Classification metrics configured.


In [13]:
# -------------------------------------------------------------------------
# REGRESSION METRICS
# -------------------------------------------------------------------------

MARGIN_LOWER_BOUND = -5.0
MARGIN_UPPER_BOUND = 5.0


def clip_margin_predictions(
    predictions,
):

    return np.clip(
        np.asarray(
            predictions,
            dtype=float,
        ),
        MARGIN_LOWER_BOUND,
        MARGIN_UPPER_BOUND,
    )


def evaluate_regression_predictions(
    y_true,
    predictions,
):

    y_true = np.asarray(
        y_true,
        dtype=float,
    )

    predictions = (
        clip_margin_predictions(
            predictions
        )
    )

    mae = mean_absolute_error(
        y_true,
        predictions,
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            predictions,
        )
    )

    if (
        np.std(
            y_true
        )
        == 0
        or
        np.std(
            predictions
        )
        == 0
    ):

        correlation = np.nan

    else:

        correlation = np.corrcoef(
            y_true,
            predictions,
        )[
            0,
            1
        ]

    return {
        "mae": float(
            mae
        ),
        "rmse": float(
            rmse
        ),
        "correlation": float(
            correlation
        ),
    }


print(
    "Regression metrics configured."
)

Regression metrics configured.


In [14]:
# -------------------------------------------------------------------------
# DATE-BLOCKED EXPANDING TEMPORAL CV
# -------------------------------------------------------------------------

N_TEMPORAL_CV_SPLITS = int(
    baseline_selection[
        "development_protocol"
    ][
        "temporal_cv_splits"
    ]
)


training_dates = np.array(
    sorted(
        training_df[
            "clean_date"
        ].unique()
    )
)


date_splitter = TimeSeriesSplit(
    n_splits=N_TEMPORAL_CV_SPLITS
)


TEMPORAL_CV_SPLITS = []

cv_summary_records = []


for fold_number, (
    train_date_indices,
    validation_date_indices,
) in enumerate(
    date_splitter.split(
        training_dates
    ),
    start=1,
):

    fold_train_dates = (
        training_dates[
            train_date_indices
        ]
    )

    fold_validation_dates = (
        training_dates[
            validation_date_indices
        ]
    )

    train_indices = np.flatnonzero(
        training_df[
            "clean_date"
        ].isin(
            fold_train_dates
        )
        .to_numpy()
    )

    validation_indices = np.flatnonzero(
        training_df[
            "clean_date"
        ].isin(
            fold_validation_dates
        )
        .to_numpy()
    )


    # ---------------------------------------------------------------------
    # TEMPORAL SAFETY
    # ---------------------------------------------------------------------

    assert len(
        train_indices
    ) > 0

    assert len(
        validation_indices
    ) > 0

    assert (
        training_df.iloc[
            train_indices
        ][
            "clean_date"
        ].max()
        <
        training_df.iloc[
            validation_indices
        ][
            "clean_date"
        ].min()
    )


    TEMPORAL_CV_SPLITS.append(
        (
            train_indices,
            validation_indices,
        )
    )


    cv_summary_records.append(
        {
            "fold": fold_number,
            "train_matches": len(
                train_indices
            ),
            "validation_matches": len(
                validation_indices
            ),
            "train_start": (
                training_df.iloc[
                    train_indices
                ][
                    "clean_date"
                ].min()
            ),
            "train_end": (
                training_df.iloc[
                    train_indices
                ][
                    "clean_date"
                ].max()
            ),
            "validation_start": (
                training_df.iloc[
                    validation_indices
                ][
                    "clean_date"
                ].min()
            ),
            "validation_end": (
                training_df.iloc[
                    validation_indices
                ][
                    "clean_date"
                ].max()
            ),
        }
    )


cv_summary_df = pd.DataFrame(
    cv_summary_records
)


print(
    f"Temporal CV folds: "
    f"{len(TEMPORAL_CV_SPLITS)}"
)

display(
    cv_summary_df
)

Temporal CV folds: 4


,fold,train_matches,validation_matches,train_start,train_end,validation_start,validation_end
0,1,43,47,2015-08-21,2015-09-22,2015-09-23,2015-10-26
1,2,90,50,2015-08-21,2015-10-26,2015-10-30,2015-12-07
2,3,140,59,2015-08-21,2015-12-07,2015-12-11,2016-01-18
3,4,199,41,2015-08-21,2016-01-18,2016-01-22,2016-02-17


In [15]:
# -------------------------------------------------------------------------
# NETWORK FEATURE SUBFAMILIES
# -------------------------------------------------------------------------

CENTRALITY_KEYWORDS = [
    "eigenvector",
    "centrality",
]


POSSESSION_FLOW_KEYWORDS = [
    "completed_passes",
    "possession_losses",
    "possession_recoveries",
]


LEAKAGE_RECOVERY_SWITCHING_KEYWORDS = [
    "leakage",
    "recovery",
    "switching",
    "finite_switching",
    "undefined_switching",
]


def columns_matching_keywords(
    columns,
    keywords,
):

    return [
        column
        for column in columns
        if any(
            keyword.lower()
            in column.lower()
            for keyword in keywords
        )
    ]


CENTRALITY_FEATURE_COLUMNS = (
    columns_matching_keywords(
        NETWORK_FEATURE_COLUMNS,
        CENTRALITY_KEYWORDS,
    )
)


POSSESSION_FLOW_FEATURE_COLUMNS = (
    columns_matching_keywords(
        NETWORK_FEATURE_COLUMNS,
        POSSESSION_FLOW_KEYWORDS,
    )
)


LEAKAGE_RECOVERY_SWITCHING_FEATURE_COLUMNS = (
    columns_matching_keywords(
        NETWORK_FEATURE_COLUMNS,
        LEAKAGE_RECOVERY_SWITCHING_KEYWORDS,
    )
)


print(
    "Network feature subfamilies:"
)

print(
    f"Centrality                  : "
    f"{len(CENTRALITY_FEATURE_COLUMNS)}"
)

print(
    f"Possession flow             : "
    f"{len(POSSESSION_FLOW_FEATURE_COLUMNS)}"
)

print(
    f"Leakage/recovery/switching  : "
    f"{len(LEAKAGE_RECOVERY_SWITCHING_FEATURE_COLUMNS)}"
)

Network feature subfamilies:
Centrality                  : 36
Possession flow             : 36
Leakage/recovery/switching  : 108


In [16]:
# -------------------------------------------------------------------------
# FEATURE ABLATION SETS
# -------------------------------------------------------------------------

FEATURE_SETS = {
    "conventional_only": (
        CONVENTIONAL_FEATURE_COLUMNS
    ),

    "network_only": (
        NETWORK_FEATURE_COLUMNS
    ),

    "conventional_plus_centrality": (
        CONVENTIONAL_FEATURE_COLUMNS
        + CENTRALITY_FEATURE_COLUMNS
    ),

    "conventional_plus_possession_flow": (
        CONVENTIONAL_FEATURE_COLUMNS
        + POSSESSION_FLOW_FEATURE_COLUMNS
    ),

    "conventional_plus_leakage_recovery_switching": (
        CONVENTIONAL_FEATURE_COLUMNS
        + LEAKAGE_RECOVERY_SWITCHING_FEATURE_COLUMNS
    ),

    "conventional_plus_all_network": (
        FULL_FEATURE_COLUMNS
    ),
}


feature_set_summary_df = pd.DataFrame(
    [
        {
            "feature_set": name,
            "n_features": len(
                columns
            ),
        }
        for name, columns
        in FEATURE_SETS.items()
    ]
)


assert (
    feature_set_summary_df[
        "n_features"
    ] > 0
).all()


display(
    feature_set_summary_df
)

,feature_set,n_features
0,conventional_only,6
1,network_only,180
2,conventional_plus_centrality,42
3,conventional_plus_possession_flow,42
4,conventional_plus_leakage_recovery_switching,114
5,conventional_plus_all_network,186


In [17]:
# -------------------------------------------------------------------------
# FIGS INTERPRETABILITY BUDGET
# -------------------------------------------------------------------------

FIGS_MAX_RULES_GRID = [
    2,
    4,
    6,
    8,
    10,
    12,
    16,
    20,
]


assert all(
    rule_count > 0
    for rule_count
    in FIGS_MAX_RULES_GRID
)


print(
    "FIGS total-split budgets:"
)

print(
    FIGS_MAX_RULES_GRID
)

print(
    "\nNumber of candidate complexities:",
    len(
        FIGS_MAX_RULES_GRID
    ),
)

FIGS total-split budgets:
[2, 4, 6, 8, 10, 12, 16, 20]

Number of candidate complexities: 8


In [18]:
# -------------------------------------------------------------------------
# FOLD-SAFE MEDIAN IMPUTATION
# -------------------------------------------------------------------------

def prepare_temporal_fold(
    dataframe,
    feature_columns,
    train_indices,
    validation_indices,
):

    X_fold_train = (
        dataframe.iloc[
            train_indices
        ][
            feature_columns
        ]
        .copy()
    )

    X_fold_validation = (
        dataframe.iloc[
            validation_indices
        ][
            feature_columns
        ]
        .copy()
    )


    imputer = SimpleImputer(
        strategy="median",
        keep_empty_features=True,
    )


    X_fold_train_imputed = (
        imputer.fit_transform(
            X_fold_train
        )
    )

    X_fold_validation_imputed = (
        imputer.transform(
            X_fold_validation
        )
    )


    assert (
        X_fold_train_imputed.shape[
            1
        ]
        ==
        len(
            feature_columns
        )
    )

    assert (
        X_fold_validation_imputed.shape[
            1
        ]
        ==
        len(
            feature_columns
        )
    )


    return (
        X_fold_train_imputed,
        X_fold_validation_imputed,
        imputer,
    )


print(
    "Fold-safe preprocessing helper configured."
)

Fold-safe preprocessing helper configured.


In [19]:
# -------------------------------------------------------------------------
# FIGS CLASSIFICATION TEMPORAL CV
# -------------------------------------------------------------------------

def evaluate_figs_classifier_cv(
    feature_columns,
    max_rules,
):

    fold_records = []


    for fold_number, (
        train_indices,
        validation_indices,
    ) in enumerate(
        TEMPORAL_CV_SPLITS,
        start=1,
    ):

        (
            X_fold_train,
            X_fold_validation,
            fold_imputer,
        ) = prepare_temporal_fold(
            dataframe=training_df,
            feature_columns=feature_columns,
            train_indices=train_indices,
            validation_indices=validation_indices,
        )


        y_fold_train = (
            training_df.iloc[
                train_indices
            ][
                "target_outcome_int"
            ]
            .to_numpy()
        )

        y_fold_validation = (
            training_df.iloc[
                validation_indices
            ][
                "target_outcome_int"
            ]
            .to_numpy()
        )


        model = FIGSClassifier(
            max_rules=max_rules,
            random_state=RANDOM_STATE,
        )


        fit_start = time.perf_counter()


        model.fit(
            X_fold_train,
            y_fold_train,
        )


        fit_seconds = (
            time.perf_counter()
            - fit_start
        )


        raw_probabilities = (
            model.predict_proba(
                X_fold_validation
            )
        )


        probabilities = (
            align_class_probabilities(
                raw_probabilities,
                model.classes_,
            )
        )


        assert probabilities.shape == (
            len(
                validation_indices
            ),
            3,
        )

        assert np.allclose(
            probabilities.sum(
                axis=1
            ),
            1.0,
            atol=1e-6,
        )


        metrics = (
            evaluate_classification_probabilities(
                y_fold_validation,
                probabilities,
            )
        )


        fold_records.append(
            {
                "fold": fold_number,
                "max_rules": max_rules,
                "fit_seconds": fit_seconds,
                **metrics,
            }
        )


    return pd.DataFrame(
        fold_records
    )

In [20]:
# -------------------------------------------------------------------------
# FIRST REAL FIGS TEMPORAL-CV TEST
# -------------------------------------------------------------------------

figs_cv_smoke_df = (
    evaluate_figs_classifier_cv(
        feature_columns=(
            CONVENTIONAL_FEATURE_COLUMNS
        ),
        max_rules=4,
    )
)


display(
    figs_cv_smoke_df
)


print(
    "\nMean RPS:",
    figs_cv_smoke_df[
        "rps"
    ].mean(),
)

,fold,max_rules,fit_seconds,rps,log_loss,brier,ece,accuracy,balanced_accuracy,macro_f1
0,1,4,0.0083,0.2597,1.1847,0.7275,0.1786,0.2979,0.2295,0.2118
1,2,4,0.0078,0.2333,1.0678,0.6457,0.1458,0.5000,0.3724,0.3367
2,3,4,0.0086,0.2194,1.0301,0.6191,0.1821,0.5763,0.3571,0.2889
3,4,4,0.0109,0.2120,1.0281,0.6173,0.1193,0.5366,0.3515,0.2873



Mean RPS: 0.23110274769021882


In [21]:
# -------------------------------------------------------------------------
# FULL FIGS CLASSIFICATION EXPERIMENT
# -------------------------------------------------------------------------

classification_cv_fold_results = []

total_experiments = (
    len(FEATURE_SETS)
    * len(FIGS_MAX_RULES_GRID)
)

experiment_counter = 0


for feature_set_name, feature_columns in (
    FEATURE_SETS.items()
):

    for max_rules in FIGS_MAX_RULES_GRID:

        experiment_counter += 1

        print(
            f"[{experiment_counter:02d}/{total_experiments}] "
            f"{feature_set_name} | max_rules={max_rules}"
        )


        fold_results_df = (
            evaluate_figs_classifier_cv(
                feature_columns=feature_columns,
                max_rules=max_rules,
            )
        )


        fold_results_df.insert(
            0,
            "feature_set",
            feature_set_name,
        )

        fold_results_df.insert(
            1,
            "n_features",
            len(feature_columns),
        )


        classification_cv_fold_results.append(
            fold_results_df
        )


figs_classification_cv_folds_df = (
    pd.concat(
        classification_cv_fold_results,
        ignore_index=True,
    )
)


print(
    "\nFull FIGS classification CV complete."
)

print(
    "Fold-level rows:",
    len(
        figs_classification_cv_folds_df
    ),
)

display(
    figs_classification_cv_folds_df.head(
        12
    )
)

[01/48] conventional_only | max_rules=2
[02/48] conventional_only | max_rules=4
[03/48] conventional_only | max_rules=6
[04/48] conventional_only | max_rules=8
[05/48] conventional_only | max_rules=10
[06/48] conventional_only | max_rules=12
[07/48] conventional_only | max_rules=16
[08/48] conventional_only | max_rules=20
[09/48] network_only | max_rules=2
[10/48] network_only | max_rules=4
[11/48] network_only | max_rules=6
[12/48] network_only | max_rules=8
[13/48] network_only | max_rules=10
[14/48] network_only | max_rules=12
[15/48] network_only | max_rules=16
[16/48] network_only | max_rules=20
[17/48] conventional_plus_centrality | max_rules=2
[18/48] conventional_plus_centrality | max_rules=4
[19/48] conventional_plus_centrality | max_rules=6
[20/48] conventional_plus_centrality | max_rules=8
[21/48] conventional_plus_centrality | max_rules=10
[22/48] conventional_plus_centrality | max_rules=12
[23/48] conventional_plus_centrality | max_rules=16
[24/48] conventional_plus_centra

,feature_set,n_features,fold,max_rules,fit_seconds,rps,log_loss,brier,ece,accuracy,balanced_accuracy,macro_f1
0,conventional_only,6,1,2,0.0059,0.2596,1.1667,0.7161,0.1470,0.2979,0.2826,0.2769
1,conventional_only,6,2,2,0.0053,0.2345,1.0914,0.6609,0.1323,0.4800,0.3446,0.2824
2,conventional_only,6,3,2,0.0057,0.2209,1.0311,0.6200,0.1918,0.5932,0.3810,0.3278
3,conventional_only,6,4,2,0.0064,0.2159,1.0430,0.6272,0.1281,0.5366,0.3515,0.2873
4,conventional_only,6,1,4,0.0087,0.2597,1.1847,0.7275,0.1786,0.2979,0.2295,0.2118
5,conventional_only,6,2,4,0.0075,0.2333,1.0678,0.6457,0.1458,0.5000,0.3724,0.3367
6,conventional_only,6,3,4,0.0080,0.2194,1.0301,0.6191,0.1821,0.5763,0.3571,0.2889
7,conventional_only,6,4,4,0.0101,0.2120,1.0281,0.6173,0.1193,0.5366,0.3515,0.2873
8,conventional_only,6,1,6,0.0109,0.2589,1.1719,0.7205,0.1881,0.2979,0.2295,0.2133
9,conventional_only,6,2,6,0.0110,0.2284,1.0498,0.6350,0.1245,0.5000,0.3724,0.3367


In [22]:
# -------------------------------------------------------------------------
# AGGREGATE FIGS CLASSIFICATION CV RESULTS
# -------------------------------------------------------------------------

figs_classification_cv_summary_df = (
    figs_classification_cv_folds_df
    .groupby(
        [
            "feature_set",
            "n_features",
            "max_rules",
        ],
        as_index=False,
    )
    .agg(
        mean_rps=(
            "rps",
            "mean",
        ),
        std_rps=(
            "rps",
            "std",
        ),
        mean_log_loss=(
            "log_loss",
            "mean",
        ),
        mean_brier=(
            "brier",
            "mean",
        ),
        mean_ece=(
            "ece",
            "mean",
        ),
        mean_accuracy=(
            "accuracy",
            "mean",
        ),
        mean_balanced_accuracy=(
            "balanced_accuracy",
            "mean",
        ),
        mean_macro_f1=(
            "macro_f1",
            "mean",
        ),
        mean_fit_seconds=(
            "fit_seconds",
            "mean",
        ),
    )
)


figs_classification_cv_summary_df[
    "se_rps"
] = (
    figs_classification_cv_summary_df[
        "std_rps"
    ]
    / np.sqrt(
        N_TEMPORAL_CV_SPLITS
    )
)


figs_classification_cv_summary_df = (
    figs_classification_cv_summary_df
    .sort_values(
        [
            "feature_set",
            "max_rules",
        ]
    )
    .reset_index(drop=True)
)


assert len(
    figs_classification_cv_summary_df
) == (
    len(FEATURE_SETS)
    * len(FIGS_MAX_RULES_GRID)
)


display(
    figs_classification_cv_summary_df
)

,feature_set,n_features,max_rules,mean_rps,std_rps,mean_log_loss,mean_brier,mean_ece,mean_accuracy,mean_balanced_accuracy,mean_macro_f1,mean_fit_seconds,se_rps
0,conventional_only,6,2,0.2327,0.0196,1.0831,0.6560,0.1498,0.4769,0.3399,0.2936,0.0058,0.0098
1,conventional_only,6,4,0.2311,0.0210,1.0777,0.6524,0.1565,0.4777,0.3276,0.2812,0.0086,0.0105
2,conventional_only,6,6,0.2299,0.0207,1.0713,0.6487,0.1531,0.4795,0.3369,0.2978,0.0125,0.0103
3,conventional_only,6,8,0.2275,0.0221,1.0577,0.6369,0.1666,0.5155,0.3830,0.3556,0.0172,0.0110
4,conventional_only,6,10,0.2340,0.0270,1.0858,0.6560,0.1561,0.4783,0.3779,0.3535,0.0220,0.0135
5,conventional_only,6,12,0.2328,0.0299,1.0842,0.6552,0.1608,0.4914,0.3982,0.3759,0.0264,0.0150
6,conventional_only,6,16,0.2293,0.0370,1.0726,0.6472,0.1685,0.4834,0.4485,0.4128,0.0380,0.0185
7,conventional_only,6,20,0.2305,0.0358,1.0751,0.6470,0.1248,0.4892,0.4580,0.4382,0.0512,0.0179
8,conventional_plus_all_network,186,2,0.2148,0.0096,1.0282,0.6165,0.1364,0.4769,0.4171,0.3916,0.0091,0.0048
9,conventional_plus_all_network,186,4,0.2246,0.0122,1.0597,0.6388,0.1016,0.4792,0.3901,0.3848,0.0157,0.0061


In [23]:
# -------------------------------------------------------------------------
# SAVE CLASSIFICATION CV RESULTS
# -------------------------------------------------------------------------

FIGS_CLASSIFICATION_CV_FOLDS_PATH = (
    FIGS_RESULTS_DIR
    / "figs_classification_cv_fold_results.csv"
)

FIGS_CLASSIFICATION_CV_SUMMARY_PATH = (
    FIGS_RESULTS_DIR
    / "figs_classification_cv_summary.csv"
)


figs_classification_cv_folds_df.to_csv(
    FIGS_CLASSIFICATION_CV_FOLDS_PATH,
    index=False,
)

figs_classification_cv_summary_df.to_csv(
    FIGS_CLASSIFICATION_CV_SUMMARY_PATH,
    index=False,
)


print(
    "Classification CV results saved."
)

print(
    FIGS_CLASSIFICATION_CV_FOLDS_PATH
)

print(
    FIGS_CLASSIFICATION_CV_SUMMARY_PATH
)

Classification CV results saved.
../data/processed/figs_results/figs_classification_cv_fold_results.csv
../data/processed/figs_results/figs_classification_cv_summary.csv


In [24]:
# -------------------------------------------------------------------------
# SELECT BEST COMPLEXITY PER FEATURE SET
# -------------------------------------------------------------------------

best_classification_complexity_records = []


for feature_set_name in FEATURE_SETS:

    subset = (
        figs_classification_cv_summary_df.loc[
            figs_classification_cv_summary_df[
                "feature_set"
            ].eq(
                feature_set_name
            )
        ]
        .sort_values(
            [
                "mean_rps",
                "max_rules",
            ],
            ascending=[
                True,
                True,
            ],
        )
        .reset_index(drop=True)
    )


    best_row = subset.iloc[
        0
    ]


    best_classification_complexity_records.append(
        best_row.to_dict()
    )


figs_classification_best_complexity_df = (
    pd.DataFrame(
        best_classification_complexity_records
    )
    .sort_values(
        "mean_rps",
        ascending=True,
    )
    .reset_index(drop=True)
)


display(
    figs_classification_best_complexity_df[
        [
            "feature_set",
            "n_features",
            "max_rules",
            "mean_rps",
            "std_rps",
            "mean_log_loss",
            "mean_brier",
            "mean_accuracy",
            "mean_macro_f1",
        ]
    ]
)

,feature_set,n_features,max_rules,mean_rps,std_rps,mean_log_loss,mean_brier,mean_accuracy,mean_macro_f1
0,conventional_plus_all_network,186,2,0.2148,0.0096,1.0282,0.6165,0.4769,0.3916
1,conventional_plus_leakage_recovery_switching,114,2,0.2158,0.0090,1.0331,0.6199,0.4609,0.3736
2,network_only,180,2,0.2163,0.0113,1.0325,0.6194,0.4615,0.3778
3,conventional_plus_possession_flow,42,2,0.2208,0.0095,1.0453,0.6296,0.5000,0.2507
4,conventional_only,6,8,0.2275,0.0221,1.0577,0.6369,0.5155,0.3556
5,conventional_plus_centrality,42,2,0.2330,0.0146,1.0776,0.6530,0.4349,0.2617


In [25]:
# -------------------------------------------------------------------------
# NETWORK ABLATION EFFECT RELATIVE TO CONVENTIONAL FIGS
# -------------------------------------------------------------------------

conventional_reference_rps = float(
    figs_classification_best_complexity_df.loc[
        figs_classification_best_complexity_df[
            "feature_set"
        ].eq(
            "conventional_only"
        ),
        "mean_rps",
    ].iloc[
        0
    ]
)


figs_classification_ablation_df = (
    figs_classification_best_complexity_df[
        [
            "feature_set",
            "n_features",
            "max_rules",
            "mean_rps",
            "std_rps",
        ]
    ]
    .copy()
)


figs_classification_ablation_df[
    "rps_difference_vs_conventional"
] = (
    figs_classification_ablation_df[
        "mean_rps"
    ]
    - conventional_reference_rps
)


figs_classification_ablation_df[
    "relative_rps_change_pct"
] = (
    100.0
    * figs_classification_ablation_df[
        "rps_difference_vs_conventional"
    ]
    / conventional_reference_rps
)


figs_classification_ablation_df[
    "improves_over_conventional"
] = (
    figs_classification_ablation_df[
        "rps_difference_vs_conventional"
    ]
    < 0
)


figs_classification_ablation_df = (
    figs_classification_ablation_df
    .sort_values(
        "mean_rps",
        ascending=True,
    )
    .reset_index(drop=True)
)


display(
    figs_classification_ablation_df
)

,feature_set,n_features,max_rules,mean_rps,std_rps,rps_difference_vs_conventional,relative_rps_change_pct,improves_over_conventional
0,conventional_plus_all_network,186,2,0.2148,0.0096,-0.0127,-5.5922,True
1,conventional_plus_leakage_recovery_switching,114,2,0.2158,0.0090,-0.0117,-5.1460,True
2,network_only,180,2,0.2163,0.0113,-0.0112,-4.9364,True
3,conventional_plus_possession_flow,42,2,0.2208,0.0095,-0.0067,-2.9436,True
4,conventional_only,6,8,0.2275,0.0221,0.0000,0.0000,False
5,conventional_plus_centrality,42,2,0.2330,0.0146,0.0054,2.3940,False


In [26]:
# -------------------------------------------------------------------------
# CV-SELECTED FIGS CLASSIFICATION CANDIDATE
# -------------------------------------------------------------------------

selected_classification_cv_row = (
    figs_classification_best_complexity_df
    .sort_values(
        [
            "mean_rps",
            "max_rules",
            "n_features",
        ],
        ascending=[
            True,
            True,
            True,
        ],
    )
    .iloc[
        0
    ]
)


SELECTED_FIGS_CLASS_FEATURE_SET = (
    selected_classification_cv_row[
        "feature_set"
    ]
)

SELECTED_FIGS_CLASS_MAX_RULES = int(
    selected_classification_cv_row[
        "max_rules"
    ]
)


print(
    "CV-selected FIGS classification candidate:"
)

print(
    "Feature set :",
    SELECTED_FIGS_CLASS_FEATURE_SET,
)

print(
    "Features    :",
    len(
        FEATURE_SETS[
            SELECTED_FIGS_CLASS_FEATURE_SET
        ]
    ),
)

print(
    "max_rules   :",
    SELECTED_FIGS_CLASS_MAX_RULES,
)

print(
    "Mean CV RPS :",
    float(
        selected_classification_cv_row[
            "mean_rps"
        ]
    ),
)

CV-selected FIGS classification candidate:
Feature set : conventional_plus_all_network
Features    : 186
max_rules   : 2
Mean CV RPS : 0.21482185073765117


In [27]:
# -------------------------------------------------------------------------
# FIT / VALIDATE A LOCKED FIGS CLASSIFIER
# -------------------------------------------------------------------------

def fit_validate_figs_classifier(
    feature_columns,
    max_rules,
):

    X_train = (
        training_df[
            feature_columns
        ]
        .copy()
    )

    X_validation = (
        validation_df[
            feature_columns
        ]
        .copy()
    )


    imputer = SimpleImputer(
        strategy="median",
        keep_empty_features=True,
    )


    X_train_imputed = (
        imputer.fit_transform(
            X_train
        )
    )

    X_validation_imputed = (
        imputer.transform(
            X_validation
        )
    )


    model = FIGSClassifier(
        max_rules=max_rules,
        random_state=RANDOM_STATE,
    )


    fit_start = time.perf_counter()


    model.fit(
        X_train_imputed,
        y_class_train,
    )


    fit_seconds = (
        time.perf_counter()
        - fit_start
    )


    raw_probabilities = (
        model.predict_proba(
            X_validation_imputed
        )
    )


    probabilities = (
        align_class_probabilities(
            raw_probabilities,
            model.classes_,
        )
    )


    assert probabilities.shape == (
        len(
            validation_df
        ),
        3,
    )

    assert np.isfinite(
        probabilities
    ).all()

    assert np.allclose(
        probabilities.sum(
            axis=1
        ),
        1.0,
        atol=1e-6,
    )


    metrics = (
        evaluate_classification_probabilities(
            y_class_validation,
            probabilities,
        )
    )


    return {
        "model": model,
        "imputer": imputer,
        "probabilities": probabilities,
        "fit_seconds": fit_seconds,
        "metrics": metrics,
    }


print(
    "Locked FIGS validation helper configured."
)

Locked FIGS validation helper configured.


In [28]:
# -------------------------------------------------------------------------
# VALIDATE CV-SELECTED FEATURE ABLATIONS
# -------------------------------------------------------------------------

classification_validation_records = []

classification_validation_fits = {}


for _, cv_row in (
    figs_classification_best_complexity_df
    .iterrows()
):

    feature_set_name = (
        cv_row[
            "feature_set"
        ]
    )

    max_rules = int(
        cv_row[
            "max_rules"
        ]
    )

    feature_columns = (
        FEATURE_SETS[
            feature_set_name
        ]
    )


    print(
        f"Validating: "
        f"{feature_set_name} | "
        f"max_rules={max_rules}"
    )


    result = (
        fit_validate_figs_classifier(
            feature_columns=feature_columns,
            max_rules=max_rules,
        )
    )


    classification_validation_fits[
        feature_set_name
    ] = result


    classification_validation_records.append(
        {
            "feature_set": feature_set_name,
            "n_features": len(
                feature_columns
            ),
            "max_rules": max_rules,
            "cv_mean_rps": float(
                cv_row[
                    "mean_rps"
                ]
            ),
            "fit_seconds": (
                result[
                    "fit_seconds"
                ]
            ),
            **result[
                "metrics"
            ],
        }
    )


figs_classification_validation_df = (
    pd.DataFrame(
        classification_validation_records
    )
    .sort_values(
        "rps",
        ascending=True,
    )
    .reset_index(drop=True)
)


display(
    figs_classification_validation_df
)

Validating: conventional_plus_all_network | max_rules=2
Validating: conventional_plus_leakage_recovery_switching | max_rules=2
Validating: network_only | max_rules=2
Validating: conventional_plus_possession_flow | max_rules=2
Validating: conventional_only | max_rules=8
Validating: conventional_plus_centrality | max_rules=2


,feature_set,n_features,max_rules,cv_mean_rps,fit_seconds,rps,log_loss,brier,ece,accuracy,balanced_accuracy,macro_f1
0,conventional_plus_possession_flow,42,2,0.2208,0.0066,0.2180,1.0372,0.6228,0.0796,0.4754,0.3968,0.3099
1,conventional_plus_all_network,186,2,0.2148,0.0304,0.2181,1.0415,0.6260,0.0523,0.4590,0.3810,0.2827
2,conventional_plus_leakage_recovery_switching,114,2,0.2158,0.0333,0.2181,1.0415,0.6260,0.0523,0.4590,0.3810,0.2827
3,network_only,180,2,0.2163,0.0172,0.2181,1.0415,0.6260,0.0523,0.4590,0.3810,0.2827
4,conventional_only,6,8,0.2275,0.0215,0.2219,1.0459,0.6298,0.0418,0.4590,0.4076,0.3649
5,conventional_plus_centrality,42,2,0.2330,0.0069,0.2291,1.0758,0.6503,0.0815,0.4426,0.3651,0.2564


In [29]:
# -------------------------------------------------------------------------
# VALIDATION ABLATION EFFECT
# -------------------------------------------------------------------------

validation_conventional_rps = float(
    figs_classification_validation_df.loc[
        figs_classification_validation_df[
            "feature_set"
        ].eq(
            "conventional_only"
        ),
        "rps",
    ].iloc[
        0
    ]
)


figs_classification_validation_df[
    "rps_difference_vs_conventional"
] = (
    figs_classification_validation_df[
        "rps"
    ]
    - validation_conventional_rps
)


figs_classification_validation_df[
    "relative_rps_change_pct"
] = (
    100.0
    * figs_classification_validation_df[
        "rps_difference_vs_conventional"
    ]
    / validation_conventional_rps
)


figs_classification_validation_df[
    "improves_over_conventional"
] = (
    figs_classification_validation_df[
        "rps_difference_vs_conventional"
    ]
    < 0
)


display(
    figs_classification_validation_df[
        [
            "feature_set",
            "max_rules",
            "cv_mean_rps",
            "rps",
            "rps_difference_vs_conventional",
            "relative_rps_change_pct",
            "log_loss",
            "brier",
            "ece",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
        ]
    ]
)

,feature_set,max_rules,cv_mean_rps,rps,rps_difference_vs_conventional,relative_rps_change_pct,log_loss,brier,ece,accuracy,balanced_accuracy,macro_f1
0,conventional_plus_possession_flow,2,0.2208,0.2180,-0.0038,-1.7324,1.0372,0.6228,0.0796,0.4754,0.3968,0.3099
1,conventional_plus_all_network,2,0.2148,0.2181,-0.0038,-1.6996,1.0415,0.6260,0.0523,0.4590,0.3810,0.2827
2,conventional_plus_leakage_recovery_switching,2,0.2158,0.2181,-0.0038,-1.6996,1.0415,0.6260,0.0523,0.4590,0.3810,0.2827
3,network_only,2,0.2163,0.2181,-0.0038,-1.6996,1.0415,0.6260,0.0523,0.4590,0.3810,0.2827
4,conventional_only,8,0.2275,0.2219,0.0000,0.0000,1.0459,0.6298,0.0418,0.4590,0.4076,0.3649
5,conventional_plus_centrality,2,0.2330,0.2291,0.0072,3.2649,1.0758,0.6503,0.0815,0.4426,0.3651,0.2564


In [30]:
# -------------------------------------------------------------------------
# LOCKED CLASSIFICATION CANDIDATE
# -------------------------------------------------------------------------

locked_classification_result = (
    classification_validation_fits[
        SELECTED_FIGS_CLASS_FEATURE_SET
    ]
)


locked_figs_classifier = (
    locked_classification_result[
        "model"
    ]
)

locked_figs_classifier_imputer = (
    locked_classification_result[
        "imputer"
    ]
)

locked_figs_classifier_probabilities = (
    locked_classification_result[
        "probabilities"
    ]
)

locked_figs_classifier_metrics = (
    locked_classification_result[
        "metrics"
    ]
)


print(
    "Locked FIGS classification model"
)

print(
    "Feature set :",
    SELECTED_FIGS_CLASS_FEATURE_SET,
)

print(
    "max_rules   :",
    SELECTED_FIGS_CLASS_MAX_RULES,
)

print(
    "\nValidation metrics:"
)

for metric_name, metric_value in (
    locked_figs_classifier_metrics.items()
):

    print(
        f"{metric_name:20s}: "
        f"{metric_value:.4f}"
    )

Locked FIGS classification model
Feature set : conventional_plus_all_network
max_rules   : 2

Validation metrics:
rps                 : 0.2181
log_loss            : 1.0415
brier               : 0.6260
ece                 : 0.0523
accuracy            : 0.4590
balanced_accuracy   : 0.3810
macro_f1            : 0.2827


In [31]:
# -------------------------------------------------------------------------
# FIGS VS NOTEBOOK 06 CLASSIFICATION BASELINES
# -------------------------------------------------------------------------

baseline_comparison_columns = [
    "model",
    "model_type",
    "rps",
    "log_loss",
    "brier",
    "ece",
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
]


baseline_comparison_df = (
    classification_baseline_results_df[
        baseline_comparison_columns
    ]
    .copy()
)


figs_comparison_row = pd.DataFrame(
    [
        {
            "model": (
                "FIGS_"
                f"{SELECTED_FIGS_CLASS_FEATURE_SET}"
            ),
            "model_type": (
                "interpretable_p2"
            ),
            **locked_figs_classifier_metrics,
        }
    ]
)


classification_model_comparison_df = (
    pd.concat(
        [
            baseline_comparison_df,
            figs_comparison_row,
        ],
        ignore_index=True,
    )
    .sort_values(
        "rps",
        ascending=True,
    )
    .reset_index(drop=True)
)


display(
    classification_model_comparison_df
)

,model,model_type,rps,log_loss,brier,ece,accuracy,balanced_accuracy,macro_f1
0,Market,external_benchmark,0.1968,0.9685,0.5723,0.0946,0.5246,0.4495,0.3826
1,RandomForest,tuned_ml,0.2106,1.0267,0.6115,0.0976,0.4426,0.3905,0.3615
2,XGBoost,tuned_ml,0.2140,1.0260,0.6173,0.1805,0.4262,0.3797,0.3601
3,GradientBoosting,tuned_ml,0.2162,1.0383,0.6259,0.0982,0.4918,0.4216,0.3661
4,FIGS_conventional_plus_all_network,interpretable_p2,0.2181,1.0415,0.6260,0.0523,0.4590,0.3810,0.2827
5,DummyPrior,reference,0.2408,1.0978,0.6671,0.0818,0.4098,0.3333,0.1938
6,SVC_RBF,tuned_ml,0.2439,1.1118,0.6742,0.0843,0.4098,0.3333,0.1938
7,LightGBM,tuned_ml,0.2536,1.2279,0.7170,0.2480,0.4590,0.4089,0.3888


In [32]:
# -------------------------------------------------------------------------
# SAVE FIGS CLASSIFICATION VALIDATION RESULTS
# -------------------------------------------------------------------------

FIGS_CLASSIFICATION_VALIDATION_PATH = (
    FIGS_RESULTS_DIR
    / "figs_classification_validation_results.csv"
)

FIGS_CLASSIFICATION_ABLATION_PATH = (
    FIGS_RESULTS_DIR
    / "figs_classification_ablation_results.csv"
)

FIGS_CLASSIFICATION_MODEL_COMPARISON_PATH = (
    FIGS_RESULTS_DIR
    / "figs_classification_model_comparison.csv"
)


figs_classification_validation_df.to_csv(
    FIGS_CLASSIFICATION_VALIDATION_PATH,
    index=False,
)

figs_classification_ablation_df.to_csv(
    FIGS_CLASSIFICATION_ABLATION_PATH,
    index=False,
)

classification_model_comparison_df.to_csv(
    FIGS_CLASSIFICATION_MODEL_COMPARISON_PATH,
    index=False,
)


print(
    "Classification validation artifacts saved."
)

Classification validation artifacts saved.


In [33]:
# -------------------------------------------------------------------------
# INSPECT LOCKED FIGS CLASSIFIER
# -------------------------------------------------------------------------

print(
    locked_figs_classifier
)

print(
    "\nAvailable fitted attributes:"
)

fitted_attribute_names = [
    name
    for name in dir(
        locked_figs_classifier
    )
    if name.endswith(
        "_"
    )
    and not name.startswith(
        "__"
    )
]


print(
    fitted_attribute_names
)

> ------------------------------
> FIGS-Fast Interpretable Greedy-Tree Sums:
> 	Predictions are made by summing the "Val" reached by traversing each tree.
> 	For classifiers, a softmax function is then applied to the sum.
> ------------------------------
X177 <= -0.034 (Tree #0 root)
	Val: 0.062 0.094 0.844 (leaf)
	X73 <= 0.082 (split)
		Val: 0.778 0.167 0.056 (leaf)
		Val: 0.278 0.323 0.399 (leaf)


Available fitted attributes:
['_repr_html_', '_repr_mimebundle_', 'classes_', 'complexity_', 'feature_importances_', 'feature_names_', 'importance_data_', 'n_features_in_', 'trees_']


In [34]:
print(
    "\nFIGS model structure:"
)

print(
    str(
        locked_figs_classifier
    )
)


FIGS model structure:
> ------------------------------
> FIGS-Fast Interpretable Greedy-Tree Sums:
> 	Predictions are made by summing the "Val" reached by traversing each tree.
> 	For classifiers, a softmax function is then applied to the sum.
> ------------------------------
X177 <= -0.034 (Tree #0 root)
	Val: 0.062 0.094 0.844 (leaf)
	X73 <= 0.082 (split)
		Val: 0.778 0.167 0.056 (leaf)
		Val: 0.278 0.323 0.399 (leaf)



In [35]:
# -------------------------------------------------------------------------
# MAP FIGS SPLIT INDICES TO FOOTBALL FEATURE NAMES
# -------------------------------------------------------------------------

locked_feature_columns = FEATURE_SETS[
    SELECTED_FIGS_CLASS_FEATURE_SET
]

print("Feature used at X177:")
print(
    locked_feature_columns[177]
)

print("\nFeature used at X73:")
print(
    locked_feature_columns[73]
)

print("\nSelected FIGS feature importances:")

feature_importance_df = (
    pd.DataFrame(
        {
            "feature": locked_feature_columns,
            "importance": (
                locked_figs_classifier.feature_importances_
            ),
        }
    )
    .loc[
        lambda df:
        df["importance"] > 0
    ]
    .sort_values(
        "importance",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    feature_importance_df
)

Feature used at X177:
diff_mean_zone_leakage_expanding_mean

Feature used at X73:
away_median_zone_leakage_roll3_mean

Selected FIGS feature importances:


,feature,importance
0,diff_mean_zone_leakage_expanding_mean,0.7204
1,away_median_zone_leakage_roll3_mean,0.2796


In [36]:
# -------------------------------------------------------------------------
# FIGS REGRESSION TEMPORAL CV
# -------------------------------------------------------------------------

def evaluate_figs_regressor_cv(
    feature_columns,
    max_rules,
):

    fold_records = []

    for fold_number, (
        train_indices,
        validation_indices,
    ) in enumerate(
        TEMPORAL_CV_SPLITS,
        start=1,
    ):

        (
            X_fold_train,
            X_fold_validation,
            fold_imputer,
        ) = prepare_temporal_fold(
            dataframe=training_df,
            feature_columns=feature_columns,
            train_indices=train_indices,
            validation_indices=validation_indices,
        )

        y_fold_train = (
            training_df.iloc[
                train_indices
            ][
                "target_margin"
            ]
            .to_numpy(
                dtype=float
            )
        )

        y_fold_validation = (
            training_df.iloc[
                validation_indices
            ][
                "target_margin"
            ]
            .to_numpy(
                dtype=float
            )
        )

        model = FIGSRegressor(
            max_rules=max_rules,
            random_state=RANDOM_STATE,
        )

        fit_start = time.perf_counter()

        model.fit(
            X_fold_train,
            y_fold_train,
        )

        fit_seconds = (
            time.perf_counter()
            - fit_start
        )

        predictions = (
            clip_margin_predictions(
                model.predict(
                    X_fold_validation
                )
            )
        )

        metrics = (
            evaluate_regression_predictions(
                y_fold_validation,
                predictions,
            )
        )

        fold_records.append(
            {
                "fold": fold_number,
                "max_rules": max_rules,
                "fit_seconds": fit_seconds,
                **metrics,
            }
        )

    return pd.DataFrame(
        fold_records
    )

In [37]:
# -------------------------------------------------------------------------
# FIRST REAL FIGS REGRESSION TEMPORAL-CV TEST
# -------------------------------------------------------------------------

figs_regression_cv_smoke_df = (
    evaluate_figs_regressor_cv(
        feature_columns=(
            CONVENTIONAL_FEATURE_COLUMNS
        ),
        max_rules=4,
    )
)

display(
    figs_regression_cv_smoke_df
)

print(
    "\nMean MAE:",
    figs_regression_cv_smoke_df[
        "mae"
    ].mean(),
)

,fold,max_rules,fit_seconds,mae,rmse,correlation
0,1,4,0.0056,2.9032,3.3668,-0.0413
1,2,4,0.0061,1.7969,2.3171,-0.1557
2,3,4,0.0063,1.3884,1.7666,0.5452
3,4,4,0.0058,1.2493,1.7312,0.4537



Mean MAE: 1.8344187839277204


In [38]:
# -------------------------------------------------------------------------
# FULL FIGS REGRESSION EXPERIMENT
# -------------------------------------------------------------------------

regression_cv_fold_results = []

total_experiments = (
    len(FEATURE_SETS)
    * len(FIGS_MAX_RULES_GRID)
)

experiment_counter = 0

for feature_set_name, feature_columns in (
    FEATURE_SETS.items()
):

    for max_rules in FIGS_MAX_RULES_GRID:

        experiment_counter += 1

        print(
            f"[{experiment_counter:02d}/{total_experiments}] "
            f"{feature_set_name} | "
            f"max_rules={max_rules}"
        )

        fold_results_df = (
            evaluate_figs_regressor_cv(
                feature_columns=feature_columns,
                max_rules=max_rules,
            )
        )

        fold_results_df.insert(
            0,
            "feature_set",
            feature_set_name,
        )

        fold_results_df.insert(
            1,
            "n_features",
            len(feature_columns),
        )

        regression_cv_fold_results.append(
            fold_results_df
        )


figs_regression_cv_folds_df = (
    pd.concat(
        regression_cv_fold_results,
        ignore_index=True,
    )
)


print(
    "\nFull FIGS regression CV complete."
)

print(
    "Fold-level rows:",
    len(
        figs_regression_cv_folds_df
    ),
)

display(
    figs_regression_cv_folds_df.head(
        12
    )
)

[01/48] conventional_only | max_rules=2
[02/48] conventional_only | max_rules=4
[03/48] conventional_only | max_rules=6
[04/48] conventional_only | max_rules=8
[05/48] conventional_only | max_rules=10
[06/48] conventional_only | max_rules=12
[07/48] conventional_only | max_rules=16
[08/48] conventional_only | max_rules=20
[09/48] network_only | max_rules=2
[10/48] network_only | max_rules=4
[11/48] network_only | max_rules=6
[12/48] network_only | max_rules=8
[13/48] network_only | max_rules=10
[14/48] network_only | max_rules=12
[15/48] network_only | max_rules=16
[16/48] network_only | max_rules=20
[17/48] conventional_plus_centrality | max_rules=2
[18/48] conventional_plus_centrality | max_rules=4
[19/48] conventional_plus_centrality | max_rules=6
[20/48] conventional_plus_centrality | max_rules=8
[21/48] conventional_plus_centrality | max_rules=10
[22/48] conventional_plus_centrality | max_rules=12
[23/48] conventional_plus_centrality | max_rules=16
[24/48] conventional_plus_centra

,feature_set,n_features,fold,max_rules,fit_seconds,mae,rmse,correlation
0,conventional_only,6,1,2,0.0026,2.1046,2.6165,-0.1775
1,conventional_only,6,2,2,0.0027,1.5903,2.1921,-0.1088
2,conventional_only,6,3,2,0.0028,1.4032,1.7822,0.5574
3,conventional_only,6,4,2,0.0032,1.1923,1.6906,0.4911
4,conventional_only,6,1,4,0.0098,2.9032,3.3668,-0.0413
5,conventional_only,6,2,4,0.0061,1.7969,2.3171,-0.1557
6,conventional_only,6,3,4,0.0067,1.3884,1.7666,0.5452
7,conventional_only,6,4,4,0.0063,1.2493,1.7312,0.4537
8,conventional_only,6,1,6,0.0078,2.9202,3.4085,-0.0159
9,conventional_only,6,2,6,0.0096,1.8678,2.4092,-0.1726


In [39]:
# -------------------------------------------------------------------------
# AGGREGATE FIGS REGRESSION CV RESULTS
# -------------------------------------------------------------------------

figs_regression_cv_summary_df = (
    figs_regression_cv_folds_df
    .groupby(
        [
            "feature_set",
            "n_features",
            "max_rules",
        ],
        as_index=False,
    )
    .agg(
        mean_mae=(
            "mae",
            "mean",
        ),
        std_mae=(
            "mae",
            "std",
        ),
        mean_rmse=(
            "rmse",
            "mean",
        ),
        mean_correlation=(
            "correlation",
            "mean",
        ),
        mean_fit_seconds=(
            "fit_seconds",
            "mean",
        ),
    )
)


figs_regression_cv_summary_df[
    "se_mae"
] = (
    figs_regression_cv_summary_df[
        "std_mae"
    ]
    / np.sqrt(
        N_TEMPORAL_CV_SPLITS
    )
)


figs_regression_cv_summary_df = (
    figs_regression_cv_summary_df
    .sort_values(
        [
            "feature_set",
            "max_rules",
        ]
    )
    .reset_index(drop=True)
)


assert len(
    figs_regression_cv_summary_df
) == 48


display(
    figs_regression_cv_summary_df
)

,feature_set,n_features,max_rules,mean_mae,std_mae,mean_rmse,mean_correlation,mean_fit_seconds,se_mae
0,conventional_only,6,2,1.5726,0.3901,2.0703,0.1905,0.0028,0.1951
1,conventional_only,6,4,1.8344,0.7495,2.2954,0.2005,0.0072,0.3747
2,conventional_only,6,6,1.8634,0.7531,2.3341,0.1590,0.0093,0.3765
3,conventional_only,6,8,1.8946,0.7473,2.3345,0.1534,0.0148,0.3736
4,conventional_only,6,10,1.9607,0.7256,2.3851,0.1356,0.0201,0.3628
5,conventional_only,6,12,2.0187,0.7071,2.4583,0.1252,0.0268,0.3536
6,conventional_only,6,16,2.1283,0.7878,2.5264,0.1153,0.0576,0.3939
7,conventional_only,6,20,2.1278,0.7103,2.5369,0.1275,0.0547,0.3552
8,conventional_plus_all_network,186,2,1.5229,0.0656,1.8997,0.3137,0.0078,0.0328
9,conventional_plus_all_network,186,4,1.4952,0.1427,1.8922,0.3214,0.0150,0.0714


In [40]:
# -------------------------------------------------------------------------
# SELECT BEST REGRESSION COMPLEXITY PER FEATURE SET
# -------------------------------------------------------------------------

best_regression_complexity_records = []


for feature_set_name in FEATURE_SETS:

    subset = (
        figs_regression_cv_summary_df.loc[
            figs_regression_cv_summary_df[
                "feature_set"
            ].eq(
                feature_set_name
            )
        ]
        .sort_values(
            [
                "mean_mae",
                "max_rules",
            ],
            ascending=[
                True,
                True,
            ],
        )
        .reset_index(drop=True)
    )

    best_regression_complexity_records.append(
        subset.iloc[
            0
        ].to_dict()
    )


figs_regression_best_complexity_df = (
    pd.DataFrame(
        best_regression_complexity_records
    )
    .sort_values(
        "mean_mae",
        ascending=True,
    )
    .reset_index(drop=True)
)


display(
    figs_regression_best_complexity_df[
        [
            "feature_set",
            "n_features",
            "max_rules",
            "mean_mae",
            "std_mae",
            "mean_rmse",
            "mean_correlation",
        ]
    ]
)

,feature_set,n_features,max_rules,mean_mae,std_mae,mean_rmse,mean_correlation
0,conventional_plus_possession_flow,42,2,1.4750,0.1454,1.8682,0.3036
1,conventional_plus_leakage_recovery_switching,114,4,1.4808,0.1913,1.8686,0.3254
2,conventional_plus_centrality,42,4,1.4862,0.1351,1.9069,0.2142
3,conventional_plus_all_network,186,4,1.4952,0.1427,1.8922,0.3214
4,network_only,180,4,1.5211,0.1806,1.9164,0.3122
5,conventional_only,6,2,1.5726,0.3901,2.0703,0.1905


In [41]:
# -------------------------------------------------------------------------
# REGRESSION NETWORK ABLATION
# -------------------------------------------------------------------------

conventional_reference_mae = float(
    figs_regression_best_complexity_df.loc[
        figs_regression_best_complexity_df[
            "feature_set"
        ].eq(
            "conventional_only"
        ),
        "mean_mae",
    ].iloc[
        0
    ]
)


figs_regression_ablation_df = (
    figs_regression_best_complexity_df[
        [
            "feature_set",
            "n_features",
            "max_rules",
            "mean_mae",
            "std_mae",
        ]
    ]
    .copy()
)


figs_regression_ablation_df[
    "mae_difference_vs_conventional"
] = (
    figs_regression_ablation_df[
        "mean_mae"
    ]
    - conventional_reference_mae
)


figs_regression_ablation_df[
    "relative_mae_change_pct"
] = (
    100.0
    * figs_regression_ablation_df[
        "mae_difference_vs_conventional"
    ]
    / conventional_reference_mae
)


figs_regression_ablation_df[
    "improves_over_conventional"
] = (
    figs_regression_ablation_df[
        "mae_difference_vs_conventional"
    ]
    < 0
)


figs_regression_ablation_df = (
    figs_regression_ablation_df
    .sort_values(
        "mean_mae"
    )
    .reset_index(drop=True)
)


display(
    figs_regression_ablation_df
)


selected_regression_cv_row = (
    figs_regression_best_complexity_df
    .sort_values(
        [
            "mean_mae",
            "max_rules",
            "n_features",
        ],
        ascending=[
            True,
            True,
            True,
        ],
    )
    .iloc[
        0
    ]
)


SELECTED_FIGS_REG_FEATURE_SET = (
    selected_regression_cv_row[
        "feature_set"
    ]
)

SELECTED_FIGS_REG_MAX_RULES = int(
    selected_regression_cv_row[
        "max_rules"
    ]
)


print(
    "\nCV-selected FIGS regression candidate:"
)

print(
    "Feature set :",
    SELECTED_FIGS_REG_FEATURE_SET,
)

print(
    "Features    :",
    len(
        FEATURE_SETS[
            SELECTED_FIGS_REG_FEATURE_SET
        ]
    ),
)

print(
    "max_rules   :",
    SELECTED_FIGS_REG_MAX_RULES,
)

print(
    "Mean CV MAE :",
    float(
        selected_regression_cv_row[
            "mean_mae"
        ]
    ),
)

,feature_set,n_features,max_rules,mean_mae,std_mae,mae_difference_vs_conventional,relative_mae_change_pct,improves_over_conventional
0,conventional_plus_possession_flow,42,2,1.4750,0.1454,-0.0976,-6.2069,True
1,conventional_plus_leakage_recovery_switching,114,4,1.4808,0.1913,-0.0918,-5.8365,True
2,conventional_plus_centrality,42,4,1.4862,0.1351,-0.0864,-5.4917,True
3,conventional_plus_all_network,186,4,1.4952,0.1427,-0.0773,-4.9178,True
4,network_only,180,4,1.5211,0.1806,-0.0515,-3.2749,True
5,conventional_only,6,2,1.5726,0.3901,0.0000,0.0000,False



CV-selected FIGS regression candidate:
Feature set : conventional_plus_possession_flow
Features    : 42
max_rules   : 2
Mean CV MAE : 1.4749653620712295


In [42]:
# -------------------------------------------------------------------------
# FIT / VALIDATE A LOCKED FIGS REGRESSOR
# -------------------------------------------------------------------------

def fit_validate_figs_regressor(
    feature_columns,
    max_rules,
):

    X_train = (
        training_df[
            feature_columns
        ]
        .copy()
    )

    X_validation = (
        validation_df[
            feature_columns
        ]
        .copy()
    )

    imputer = SimpleImputer(
        strategy="median",
        keep_empty_features=True,
    )

    X_train_imputed = (
        imputer.fit_transform(
            X_train
        )
    )

    X_validation_imputed = (
        imputer.transform(
            X_validation
        )
    )

    model = FIGSRegressor(
        max_rules=max_rules,
        random_state=RANDOM_STATE,
    )

    fit_start = time.perf_counter()

    model.fit(
        X_train_imputed,
        y_margin_train,
    )

    fit_seconds = (
        time.perf_counter()
        - fit_start
    )

    predictions = (
        clip_margin_predictions(
            model.predict(
                X_validation_imputed
            )
        )
    )

    metrics = (
        evaluate_regression_predictions(
            y_margin_validation,
            predictions,
        )
    )

    return {
        "model": model,
        "imputer": imputer,
        "predictions": predictions,
        "fit_seconds": fit_seconds,
        "metrics": metrics,
    }


print(
    "Locked FIGS regression validation helper configured."
)

Locked FIGS regression validation helper configured.


In [43]:
# -------------------------------------------------------------------------
# VALIDATE CV-SELECTED REGRESSION ABLATIONS
# -------------------------------------------------------------------------

regression_validation_records = []

regression_validation_fits = {}


for _, cv_row in (
    figs_regression_best_complexity_df
    .iterrows()
):

    feature_set_name = (
        cv_row[
            "feature_set"
        ]
    )

    max_rules = int(
        cv_row[
            "max_rules"
        ]
    )

    feature_columns = (
        FEATURE_SETS[
            feature_set_name
        ]
    )

    print(
        f"Validating: "
        f"{feature_set_name} | "
        f"max_rules={max_rules}"
    )

    result = (
        fit_validate_figs_regressor(
            feature_columns=feature_columns,
            max_rules=max_rules,
        )
    )

    regression_validation_fits[
        feature_set_name
    ] = result

    regression_validation_records.append(
        {
            "feature_set": feature_set_name,
            "n_features": len(
                feature_columns
            ),
            "max_rules": max_rules,
            "cv_mean_mae": float(
                cv_row[
                    "mean_mae"
                ]
            ),
            "fit_seconds": (
                result[
                    "fit_seconds"
                ]
            ),
            **result[
                "metrics"
            ],
        }
    )


figs_regression_validation_df = (
    pd.DataFrame(
        regression_validation_records
    )
    .sort_values(
        "mae",
        ascending=True,
    )
    .reset_index(drop=True)
)


display(
    figs_regression_validation_df
)

Validating: conventional_plus_possession_flow | max_rules=2
Validating: conventional_plus_leakage_recovery_switching | max_rules=4
Validating: conventional_plus_centrality | max_rules=4
Validating: conventional_plus_all_network | max_rules=4
Validating: network_only | max_rules=4
Validating: conventional_only | max_rules=2


,feature_set,n_features,max_rules,cv_mean_mae,fit_seconds,mae,rmse,correlation
0,conventional_plus_possession_flow,42,2,1.4750,0.0049,1.4214,1.7525,0.4639
1,conventional_plus_leakage_recovery_switching,114,4,1.4808,0.0189,1.4503,1.7857,0.4187
2,conventional_plus_all_network,186,4,1.4952,0.0209,1.4503,1.7857,0.4187
3,conventional_only,6,2,1.5726,0.0026,1.4962,1.8149,0.3969
4,conventional_plus_centrality,42,4,1.4862,0.0093,1.5174,1.9314,0.2613
5,network_only,180,4,1.5211,0.0195,1.5416,2.0350,0.2049


In [44]:
# -------------------------------------------------------------------------
# REGRESSION VALIDATION ABLATION EFFECT
# -------------------------------------------------------------------------

validation_conventional_mae = float(
    figs_regression_validation_df.loc[
        figs_regression_validation_df[
            "feature_set"
        ].eq(
            "conventional_only"
        ),
        "mae",
    ].iloc[
        0
    ]
)


figs_regression_validation_df[
    "mae_difference_vs_conventional"
] = (
    figs_regression_validation_df[
        "mae"
    ]
    - validation_conventional_mae
)


figs_regression_validation_df[
    "relative_mae_change_pct"
] = (
    100.0
    * figs_regression_validation_df[
        "mae_difference_vs_conventional"
    ]
    / validation_conventional_mae
)


figs_regression_validation_df[
    "improves_over_conventional"
] = (
    figs_regression_validation_df[
        "mae_difference_vs_conventional"
    ]
    < 0
)


display(
    figs_regression_validation_df[
        [
            "feature_set",
            "max_rules",
            "cv_mean_mae",
            "mae",
            "mae_difference_vs_conventional",
            "relative_mae_change_pct",
            "rmse",
            "correlation",
        ]
    ]
)

,feature_set,max_rules,cv_mean_mae,mae,mae_difference_vs_conventional,relative_mae_change_pct,rmse,correlation
0,conventional_plus_possession_flow,2,1.4750,1.4214,-0.0748,-5.0019,1.7525,0.4639
1,conventional_plus_leakage_recovery_switching,4,1.4808,1.4503,-0.0460,-3.0737,1.7857,0.4187
2,conventional_plus_all_network,4,1.4952,1.4503,-0.0460,-3.0737,1.7857,0.4187
3,conventional_only,2,1.5726,1.4962,0.0000,0.0000,1.8149,0.3969
4,conventional_plus_centrality,4,1.4862,1.5174,0.0212,1.4145,1.9314,0.2613
5,network_only,4,1.5211,1.5416,0.0453,3.0306,2.0350,0.2049


In [45]:
# -------------------------------------------------------------------------
# LOCKED FIGS REGRESSION CANDIDATE
# -------------------------------------------------------------------------

locked_regression_result = (
    regression_validation_fits[
        SELECTED_FIGS_REG_FEATURE_SET
    ]
)


locked_figs_regressor = (
    locked_regression_result[
        "model"
    ]
)

locked_figs_regressor_imputer = (
    locked_regression_result[
        "imputer"
    ]
)

locked_figs_regressor_predictions = (
    locked_regression_result[
        "predictions"
    ]
)

locked_figs_regressor_metrics = (
    locked_regression_result[
        "metrics"
    ]
)


print(
    "Locked FIGS regression model"
)

print(
    "Feature set :",
    SELECTED_FIGS_REG_FEATURE_SET,
)

print(
    "max_rules   :",
    SELECTED_FIGS_REG_MAX_RULES,
)

print(
    "\nValidation metrics:"
)

for metric_name, metric_value in (
    locked_figs_regressor_metrics.items()
):

    print(
        f"{metric_name:15s}: "
        f"{metric_value:.4f}"
    )

Locked FIGS regression model
Feature set : conventional_plus_possession_flow
max_rules   : 2

Validation metrics:
mae            : 1.4214
rmse           : 1.7525
correlation    : 0.4639


In [46]:
print(
    regression_baseline_results_df.columns.tolist()
)

display(
    regression_baseline_results_df
)

['model', 'model_type', 'mae', 'rmse', 'correlation']


,model,model_type,mae,rmse,correlation
0,RandomForest,tuned_ml,1.3583,1.6879,0.5355
1,LightGBM,tuned_ml,1.4080,1.7540,0.4505
2,GradientBoosting,tuned_ml,1.4108,1.7644,0.4762
3,XGBoost,tuned_ml,1.4397,1.7373,0.4657
4,KernelRidge_RBF,tuned_ml,1.4439,1.8506,0.4846
5,SVR_RBF,tuned_ml,1.4448,1.8128,0.5017
6,Nystroem_Ridge,tuned_ml,1.4858,1.8835,0.3667
7,DummyMean,reference,1.5708,1.9754,0.0000


In [47]:
# -------------------------------------------------------------------------
# FIGS VS NOTEBOOK 06 REGRESSION BASELINES
# -------------------------------------------------------------------------

regression_baseline_comparison_df = (
    regression_baseline_results_df[
        [
            "model",
            "model_type",
            "mae",
            "rmse",
            "correlation",
        ]
    ]
    .copy()
)


figs_regression_comparison_row = pd.DataFrame(
    [
        {
            "model": (
                "FIGS_"
                f"{SELECTED_FIGS_REG_FEATURE_SET}"
            ),
            "model_type": "interpretable_p2",
            **locked_figs_regressor_metrics,
        }
    ]
)


regression_model_comparison_df = (
    pd.concat(
        [
            regression_baseline_comparison_df,
            figs_regression_comparison_row,
        ],
        ignore_index=True,
    )
    .sort_values(
        "mae",
        ascending=True,
    )
    .reset_index(drop=True)
)


display(
    regression_model_comparison_df
)

,model,model_type,mae,rmse,correlation
0,RandomForest,tuned_ml,1.3583,1.6879,0.5355
1,LightGBM,tuned_ml,1.4080,1.7540,0.4505
2,GradientBoosting,tuned_ml,1.4108,1.7644,0.4762
3,FIGS_conventional_plus_possession_flow,interpretable_p2,1.4214,1.7525,0.4639
4,XGBoost,tuned_ml,1.4397,1.7373,0.4657
5,KernelRidge_RBF,tuned_ml,1.4439,1.8506,0.4846
6,SVR_RBF,tuned_ml,1.4448,1.8128,0.5017
7,Nystroem_Ridge,tuned_ml,1.4858,1.8835,0.3667
8,DummyMean,reference,1.5708,1.9754,0.0000


In [48]:
# -------------------------------------------------------------------------
# SAVE FIGS REGRESSION RESULTS
# -------------------------------------------------------------------------

FIGS_REGRESSION_CV_FOLDS_PATH = (
    FIGS_RESULTS_DIR
    / "figs_regression_cv_fold_results.csv"
)

FIGS_REGRESSION_CV_SUMMARY_PATH = (
    FIGS_RESULTS_DIR
    / "figs_regression_cv_summary.csv"
)

FIGS_REGRESSION_VALIDATION_PATH = (
    FIGS_RESULTS_DIR
    / "figs_regression_validation_results.csv"
)

FIGS_REGRESSION_ABLATION_PATH = (
    FIGS_RESULTS_DIR
    / "figs_regression_ablation_results.csv"
)

FIGS_REGRESSION_MODEL_COMPARISON_PATH = (
    FIGS_RESULTS_DIR
    / "figs_regression_model_comparison.csv"
)


figs_regression_cv_folds_df.to_csv(
    FIGS_REGRESSION_CV_FOLDS_PATH,
    index=False,
)

figs_regression_cv_summary_df.to_csv(
    FIGS_REGRESSION_CV_SUMMARY_PATH,
    index=False,
)

figs_regression_validation_df.to_csv(
    FIGS_REGRESSION_VALIDATION_PATH,
    index=False,
)

figs_regression_ablation_df.to_csv(
    FIGS_REGRESSION_ABLATION_PATH,
    index=False,
)

regression_model_comparison_df.to_csv(
    FIGS_REGRESSION_MODEL_COMPARISON_PATH,
    index=False,
)

print(
    "Regression FIGS artifacts saved."
)

Regression FIGS artifacts saved.


In [49]:
# -------------------------------------------------------------------------
# INSPECT LOCKED FIGS REGRESSOR
# -------------------------------------------------------------------------

print(
    locked_figs_regressor
)

print(
    "\nFIGS regression structure:"
)

print(
    str(
        locked_figs_regressor
    )
)


locked_regression_feature_columns = (
    FEATURE_SETS[
        SELECTED_FIGS_REG_FEATURE_SET
    ]
)


regression_feature_importance_df = (
    pd.DataFrame(
        {
            "feature": (
                locked_regression_feature_columns
            ),
            "importance": (
                locked_figs_regressor
                .feature_importances_
            ),
        }
    )
    .loc[
        lambda df:
        df[
            "importance"
        ] > 0
    ]
    .sort_values(
        "importance",
        ascending=False,
    )
    .reset_index(drop=True)
)


display(
    regression_feature_importance_df
)

> ------------------------------
> FIGS-Fast Interpretable Greedy-Tree Sums:
> 	Predictions are made by summing the "Val" reached by traversing each tree.
> 	For classifiers, a softmax function is then applied to the sum.
> ------------------------------
X36 <= 124.050 (Tree #0 root)
	X39 <= -255.105 (split)
		Val: -2.5 (leaf)
		Val: 0.289 (leaf)
	Val: 2.545 (leaf)


FIGS regression structure:
> ------------------------------
> FIGS-Fast Interpretable Greedy-Tree Sums:
> 	Predictions are made by summing the "Val" reached by traversing each tree.
> 	For classifiers, a softmax function is then applied to the sum.
> ------------------------------
X36 <= 124.050 (Tree #0 root)
	X39 <= -255.105 (split)
		Val: -2.5 (leaf)
		Val: 0.289 (leaf)
	Val: 2.545 (leaf)



,feature,importance
0,diff_completed_passes_roll10_mean,0.7744
1,diff_completed_passes_expanding_mean,0.2256


In [50]:
# -------------------------------------------------------------------------
# CART COMPLEXITY HELPERS
# -------------------------------------------------------------------------

def count_cart_splits(
    fitted_tree_model,
):
    """
    Count non-leaf nodes in a fitted sklearn CART tree.
    """

    tree = fitted_tree_model.tree_

    is_split_node = (
        tree.children_left
        != tree.children_right
    )

    return int(
        np.sum(
            is_split_node
        )
    )


print(
    "CART split-count helper configured."
)

CART split-count helper configured.


In [51]:
# -------------------------------------------------------------------------
# MATCHED-COMPLEXITY CART CLASSIFIER
# -------------------------------------------------------------------------

cart_class_feature_columns = (
    FEATURE_SETS[
        SELECTED_FIGS_CLASS_FEATURE_SET
    ]
)


cart_class_imputer = SimpleImputer(
    strategy="median",
    keep_empty_features=True,
)


X_cart_class_train = (
    cart_class_imputer.fit_transform(
        training_df[
            cart_class_feature_columns
        ]
    )
)

X_cart_class_validation = (
    cart_class_imputer.transform(
        validation_df[
            cart_class_feature_columns
        ]
    )
)


cart_classifier = DecisionTreeClassifier(
    max_leaf_nodes=(
        SELECTED_FIGS_CLASS_MAX_RULES
        + 1
    ),
    random_state=RANDOM_STATE,
)


cart_classifier.fit(
    X_cart_class_train,
    y_class_train,
)


cart_class_splits = count_cart_splits(
    cart_classifier
)


cart_class_probabilities = (
    align_class_probabilities(
        cart_classifier.predict_proba(
            X_cart_class_validation
        ),
        cart_classifier.classes_,
    )
)


cart_class_metrics = (
    evaluate_classification_probabilities(
        y_class_validation,
        cart_class_probabilities,
    )
)


print(
    "Matched CART classification"
)

print(
    "Maximum split budget:",
    SELECTED_FIGS_CLASS_MAX_RULES,
)

print(
    "Actual CART splits  :",
    cart_class_splits,
)

print(
    "\nValidation metrics:"
)

for metric_name, metric_value in (
    cart_class_metrics.items()
):

    print(
        f"{metric_name:20s}: "
        f"{metric_value:.4f}"
    )

Matched CART classification
Maximum split budget: 2
Actual CART splits  : 2

Validation metrics:
rps                 : 0.2198
log_loss            : 1.0531
brier               : 0.6339
ece                 : 0.0802
accuracy            : 0.4590
balanced_accuracy   : 0.3810
macro_f1            : 0.2827


In [52]:
# -------------------------------------------------------------------------
# FIGS VS MATCHED CART — CLASSIFICATION
# -------------------------------------------------------------------------

figs_vs_cart_classification_df = pd.DataFrame(
    [
        {
            "model": "FIGS",
            "feature_set": (
                SELECTED_FIGS_CLASS_FEATURE_SET
            ),
            "split_budget": (
                SELECTED_FIGS_CLASS_MAX_RULES
            ),
            "actual_splits": (
                SELECTED_FIGS_CLASS_MAX_RULES
            ),
            **locked_figs_classifier_metrics,
        },
        {
            "model": "CART",
            "feature_set": (
                SELECTED_FIGS_CLASS_FEATURE_SET
            ),
            "split_budget": (
                SELECTED_FIGS_CLASS_MAX_RULES
            ),
            "actual_splits": (
                cart_class_splits
            ),
            **cart_class_metrics,
        },
    ]
).sort_values(
    "rps",
    ascending=True,
).reset_index(
    drop=True
)


display(
    figs_vs_cart_classification_df
)

,model,feature_set,split_budget,actual_splits,rps,log_loss,brier,ece,accuracy,balanced_accuracy,macro_f1
0,FIGS,conventional_plus_all_network,2,2,0.2181,1.0415,0.6260,0.0523,0.4590,0.3810,0.2827
1,CART,conventional_plus_all_network,2,2,0.2198,1.0531,0.6339,0.0802,0.4590,0.3810,0.2827


In [53]:
# -------------------------------------------------------------------------
# MATCHED-COMPLEXITY CART REGRESSOR
# -------------------------------------------------------------------------

cart_reg_feature_columns = (
    FEATURE_SETS[
        SELECTED_FIGS_REG_FEATURE_SET
    ]
)


cart_reg_imputer = SimpleImputer(
    strategy="median",
    keep_empty_features=True,
)


X_cart_reg_train = (
    cart_reg_imputer.fit_transform(
        training_df[
            cart_reg_feature_columns
        ]
    )
)

X_cart_reg_validation = (
    cart_reg_imputer.transform(
        validation_df[
            cart_reg_feature_columns
        ]
    )
)


cart_regressor = DecisionTreeRegressor(
    max_leaf_nodes=(
        SELECTED_FIGS_REG_MAX_RULES
        + 1
    ),
    random_state=RANDOM_STATE,
)


cart_regressor.fit(
    X_cart_reg_train,
    y_margin_train,
)


cart_reg_splits = count_cart_splits(
    cart_regressor
)


cart_reg_predictions = (
    clip_margin_predictions(
        cart_regressor.predict(
            X_cart_reg_validation
        )
    )
)


cart_reg_metrics = (
    evaluate_regression_predictions(
        y_margin_validation,
        cart_reg_predictions,
    )
)


print(
    "Matched CART regression"
)

print(
    "Maximum split budget:",
    SELECTED_FIGS_REG_MAX_RULES,
)

print(
    "Actual CART splits  :",
    cart_reg_splits,
)

print(
    "\nValidation metrics:"
)

for metric_name, metric_value in (
    cart_reg_metrics.items()
):

    print(
        f"{metric_name:15s}: "
        f"{metric_value:.4f}"
    )

Matched CART regression
Maximum split budget: 2
Actual CART splits  : 2

Validation metrics:
mae            : 1.4214
rmse           : 1.7525
correlation    : 0.4639


In [54]:
# -------------------------------------------------------------------------
# FIGS VS MATCHED CART — REGRESSION
# -------------------------------------------------------------------------

figs_vs_cart_regression_df = pd.DataFrame(
    [
        {
            "model": "FIGS",
            "feature_set": (
                SELECTED_FIGS_REG_FEATURE_SET
            ),
            "split_budget": (
                SELECTED_FIGS_REG_MAX_RULES
            ),
            "actual_splits": (
                SELECTED_FIGS_REG_MAX_RULES
            ),
            **locked_figs_regressor_metrics,
        },
        {
            "model": "CART",
            "feature_set": (
                SELECTED_FIGS_REG_FEATURE_SET
            ),
            "split_budget": (
                SELECTED_FIGS_REG_MAX_RULES
            ),
            "actual_splits": (
                cart_reg_splits
            ),
            **cart_reg_metrics,
        },
    ]
).sort_values(
    "mae",
    ascending=True,
).reset_index(
    drop=True
)


display(
    figs_vs_cart_regression_df
)

,model,feature_set,split_budget,actual_splits,mae,rmse,correlation
0,FIGS,conventional_plus_possession_flow,2,2,1.4214,1.7525,0.4639
1,CART,conventional_plus_possession_flow,2,2,1.4214,1.7525,0.4639


In [55]:
# -------------------------------------------------------------------------
# SAVE MATCHED-COMPLEXITY COMPARISONS
# -------------------------------------------------------------------------

FIGS_VS_CART_CLASSIFICATION_PATH = (
    FIGS_RESULTS_DIR
    / "figs_vs_cart_classification.csv"
)

FIGS_VS_CART_REGRESSION_PATH = (
    FIGS_RESULTS_DIR
    / "figs_vs_cart_regression.csv"
)


figs_vs_cart_classification_df.to_csv(
    FIGS_VS_CART_CLASSIFICATION_PATH,
    index=False,
)

figs_vs_cart_regression_df.to_csv(
    FIGS_VS_CART_REGRESSION_PATH,
    index=False,
)


print(
    "Matched-complexity CART comparisons saved."
)

Matched-complexity CART comparisons saved.


In [56]:
# -------------------------------------------------------------------------
# FIGS SELECTED-FEATURE EXTRACTION
# -------------------------------------------------------------------------

def get_figs_selected_features(
    fitted_model,
    feature_columns,
):

    importances = np.asarray(
        fitted_model.feature_importances_,
        dtype=float,
    )

    selected_indices = np.flatnonzero(
        importances > 0
    )

    records = []

    for feature_index in selected_indices:

        records.append(
            {
                "feature_index": int(
                    feature_index
                ),
                "feature": feature_columns[
                    feature_index
                ],
                "importance": float(
                    importances[
                        feature_index
                    ]
                ),
            }
        )

    return (
        pd.DataFrame(
            records
        )
        .sort_values(
            "importance",
            ascending=False,
        )
        .reset_index(drop=True)
    )


print(
    "FIGS feature-extraction helper configured."
)

FIGS feature-extraction helper configured.


In [57]:
# -------------------------------------------------------------------------
# CLASSIFICATION FIGS TEMPORAL STRUCTURAL STABILITY
# -------------------------------------------------------------------------

classification_stability_records = []


locked_class_feature_columns = (
    FEATURE_SETS[
        SELECTED_FIGS_CLASS_FEATURE_SET
    ]
)


for fold_number, (
    train_indices,
    validation_indices,
) in enumerate(
    TEMPORAL_CV_SPLITS,
    start=1,
):

    (
        X_fold_train,
        X_fold_validation,
        fold_imputer,
    ) = prepare_temporal_fold(
        dataframe=training_df,
        feature_columns=(
            locked_class_feature_columns
        ),
        train_indices=train_indices,
        validation_indices=validation_indices,
    )

    y_fold_train = (
        training_df.iloc[
            train_indices
        ][
            "target_outcome_int"
        ]
        .to_numpy()
    )

    fold_model = FIGSClassifier(
        max_rules=(
            SELECTED_FIGS_CLASS_MAX_RULES
        ),
        random_state=RANDOM_STATE,
    )

    fold_model.fit(
        X_fold_train,
        y_fold_train,
    )

    fold_selected_df = (
        get_figs_selected_features(
            fitted_model=fold_model,
            feature_columns=(
                locked_class_feature_columns
            ),
        )
    )

    for _, selected_row in (
        fold_selected_df.iterrows()
    ):

        classification_stability_records.append(
            {
                "fold": fold_number,
                "train_matches": len(
                    train_indices
                ),
                "feature": (
                    selected_row[
                        "feature"
                    ]
                ),
                "importance": (
                    selected_row[
                        "importance"
                    ]
                ),
            }
        )


figs_classification_stability_df = (
    pd.DataFrame(
        classification_stability_records
    )
)


display(
    figs_classification_stability_df
)

,fold,train_matches,feature,importance
0,1,43,diff_mean_zone_leakage_roll3_mean,0.6186
1,1,43,home_median_zone_leakage_roll10_mean,0.3814
2,2,90,home_median_zone_leakage_roll10_mean,0.5778
3,2,90,diff_maximum_zone_leakage_roll10_mean,0.4222
4,3,140,home_median_zone_leakage_roll10_mean,0.6993
5,3,140,away_median_zone_leakage_roll3_mean,0.3007
6,4,199,diff_mean_zone_leakage_roll5_mean,0.7650
7,4,199,diff_median_zone_leakage_expanding_mean,0.2350


In [58]:
# -------------------------------------------------------------------------
# CLASSIFICATION FEATURE STABILITY SUMMARY
# -------------------------------------------------------------------------

classification_feature_stability_summary_df = (
    figs_classification_stability_df
    .groupby(
        "feature",
        as_index=False,
    )
    .agg(
        folds_selected=(
            "fold",
            "nunique",
        ),
        mean_importance=(
            "importance",
            "mean",
        ),
    )
)


classification_feature_stability_summary_df[
    "selection_rate"
] = (
    classification_feature_stability_summary_df[
        "folds_selected"
    ]
    / N_TEMPORAL_CV_SPLITS
)


classification_feature_stability_summary_df = (
    classification_feature_stability_summary_df
    .sort_values(
        [
            "folds_selected",
            "mean_importance",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)


display(
    classification_feature_stability_summary_df
)

,feature,folds_selected,mean_importance,selection_rate
0,home_median_zone_leakage_roll10_mean,3,0.5528,0.7500
1,diff_mean_zone_leakage_roll5_mean,1,0.7650,0.2500
2,diff_mean_zone_leakage_roll3_mean,1,0.6186,0.2500
3,diff_maximum_zone_leakage_roll10_mean,1,0.4222,0.2500
4,away_median_zone_leakage_roll3_mean,1,0.3007,0.2500
5,diff_median_zone_leakage_expanding_mean,1,0.2350,0.2500


In [59]:
# -------------------------------------------------------------------------
# REGRESSION FIGS TEMPORAL STRUCTURAL STABILITY
# -------------------------------------------------------------------------

regression_stability_records = []


locked_reg_feature_columns = (
    FEATURE_SETS[
        SELECTED_FIGS_REG_FEATURE_SET
    ]
)


for fold_number, (
    train_indices,
    validation_indices,
) in enumerate(
    TEMPORAL_CV_SPLITS,
    start=1,
):

    (
        X_fold_train,
        X_fold_validation,
        fold_imputer,
    ) = prepare_temporal_fold(
        dataframe=training_df,
        feature_columns=(
            locked_reg_feature_columns
        ),
        train_indices=train_indices,
        validation_indices=validation_indices,
    )

    y_fold_train = (
        training_df.iloc[
            train_indices
        ][
            "target_margin"
        ]
        .to_numpy(
            dtype=float
        )
    )

    fold_model = FIGSRegressor(
        max_rules=(
            SELECTED_FIGS_REG_MAX_RULES
        ),
        random_state=RANDOM_STATE,
    )

    fold_model.fit(
        X_fold_train,
        y_fold_train,
    )

    fold_selected_df = (
        get_figs_selected_features(
            fitted_model=fold_model,
            feature_columns=(
                locked_reg_feature_columns
            ),
        )
    )

    for _, selected_row in (
        fold_selected_df.iterrows()
    ):

        regression_stability_records.append(
            {
                "fold": fold_number,
                "train_matches": len(
                    train_indices
                ),
                "feature": (
                    selected_row[
                        "feature"
                    ]
                ),
                "importance": (
                    selected_row[
                        "importance"
                    ]
                ),
            }
        )


figs_regression_stability_df = (
    pd.DataFrame(
        regression_stability_records
    )
)


display(
    figs_regression_stability_df
)

,fold,train_matches,feature,importance
0,1,43,diff_completed_passes_roll5_mean,0.5684
1,1,43,diff_completed_passes_roll3_mean,0.4316
2,2,90,diff_completed_passes_roll5_mean,0.6137
3,2,90,home_completed_passes_expanding_mean,0.3863
4,3,140,home_completed_passes_roll10_mean,0.6095
5,3,140,away_completed_passes_roll3_mean,0.3905
6,4,199,diff_completed_passes_expanding_mean,0.7801
7,4,199,away_completed_passes_roll10_mean,0.2199


In [60]:
# -------------------------------------------------------------------------
# REGRESSION FEATURE STABILITY SUMMARY
# -------------------------------------------------------------------------

regression_feature_stability_summary_df = (
    figs_regression_stability_df
    .groupby(
        "feature",
        as_index=False,
    )
    .agg(
        folds_selected=(
            "fold",
            "nunique",
        ),
        mean_importance=(
            "importance",
            "mean",
        ),
    )
)


regression_feature_stability_summary_df[
    "selection_rate"
] = (
    regression_feature_stability_summary_df[
        "folds_selected"
    ]
    / N_TEMPORAL_CV_SPLITS
)


regression_feature_stability_summary_df = (
    regression_feature_stability_summary_df
    .sort_values(
        [
            "folds_selected",
            "mean_importance",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)


display(
    regression_feature_stability_summary_df
)

,feature,folds_selected,mean_importance,selection_rate
0,diff_completed_passes_roll5_mean,2,0.5911,0.5000
1,diff_completed_passes_expanding_mean,1,0.7801,0.2500
2,home_completed_passes_roll10_mean,1,0.6095,0.2500
3,diff_completed_passes_roll3_mean,1,0.4316,0.2500
4,away_completed_passes_roll3_mean,1,0.3905,0.2500
5,home_completed_passes_expanding_mean,1,0.3863,0.2500
6,away_completed_passes_roll10_mean,1,0.2199,0.2500


In [61]:
# -------------------------------------------------------------------------
# CONCEPT-LEVEL TEMPORAL STABILITY
# -------------------------------------------------------------------------

def classify_network_concept(
    feature_name,
):

    feature_name_lower = (
        feature_name.lower()
    )

    if "leakage" in feature_name_lower:
        return "leakage"

    if "recovery" in feature_name_lower:
        return "recovery"

    if "switching" in feature_name_lower:
        return "switching"

    if "completed_passes" in feature_name_lower:
        return "completed_passes"

    if "centrality" in feature_name_lower:
        return "centrality"

    if "eigenvector" in feature_name_lower:
        return "centrality"

    if "possession_losses" in feature_name_lower:
        return "possession_losses"

    if "possession_recoveries" in feature_name_lower:
        return "possession_recoveries"

    return "other"


classification_concept_stability_df = (
    figs_classification_stability_df
    .copy()
)

classification_concept_stability_df[
    "concept"
] = (
    classification_concept_stability_df[
        "feature"
    ]
    .map(
        classify_network_concept
    )
)


regression_concept_stability_df = (
    figs_regression_stability_df
    .copy()
)

regression_concept_stability_df[
    "concept"
] = (
    regression_concept_stability_df[
        "feature"
    ]
    .map(
        classify_network_concept
    )
)


classification_concept_summary_df = (
    classification_concept_stability_df
    .groupby(
        "concept",
        as_index=False,
    )
    .agg(
        selections=(
            "feature",
            "size",
        ),
        folds_present=(
            "fold",
            "nunique",
        ),
        mean_importance=(
            "importance",
            "mean",
        ),
    )
    .sort_values(
        [
            "folds_present",
            "selections",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)


regression_concept_summary_df = (
    regression_concept_stability_df
    .groupby(
        "concept",
        as_index=False,
    )
    .agg(
        selections=(
            "feature",
            "size",
        ),
        folds_present=(
            "fold",
            "nunique",
        ),
        mean_importance=(
            "importance",
            "mean",
        ),
    )
    .sort_values(
        [
            "folds_present",
            "selections",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "Classification concept stability:"
)

display(
    classification_concept_summary_df
)


print(
    "\nRegression concept stability:"
)

display(
    regression_concept_summary_df
)

Classification concept stability:


,concept,selections,folds_present,mean_importance
0,leakage,8,4,0.5000



Regression concept stability:


,concept,selections,folds_present,mean_importance
0,completed_passes,8,4,0.5000


In [62]:
# -------------------------------------------------------------------------
# SAVE TEMPORAL STABILITY RESULTS
# -------------------------------------------------------------------------

FIGS_CLASSIFICATION_STABILITY_PATH = (
    FIGS_RESULTS_DIR
    / "figs_classification_temporal_stability.csv"
)

FIGS_CLASSIFICATION_STABILITY_SUMMARY_PATH = (
    FIGS_RESULTS_DIR
    / "figs_classification_temporal_stability_summary.csv"
)

FIGS_REGRESSION_STABILITY_PATH = (
    FIGS_RESULTS_DIR
    / "figs_regression_temporal_stability.csv"
)

FIGS_REGRESSION_STABILITY_SUMMARY_PATH = (
    FIGS_RESULTS_DIR
    / "figs_regression_temporal_stability_summary.csv"
)


figs_classification_stability_df.to_csv(
    FIGS_CLASSIFICATION_STABILITY_PATH,
    index=False,
)

classification_feature_stability_summary_df.to_csv(
    FIGS_CLASSIFICATION_STABILITY_SUMMARY_PATH,
    index=False,
)

figs_regression_stability_df.to_csv(
    FIGS_REGRESSION_STABILITY_PATH,
    index=False,
)

regression_feature_stability_summary_df.to_csv(
    FIGS_REGRESSION_STABILITY_SUMMARY_PATH,
    index=False,
)


print(
    "Temporal stability artifacts saved."
)

Temporal stability artifacts saved.


In [63]:
# -------------------------------------------------------------------------
# FINAL DEVELOPMENT LOCK — BEFORE OPENING TEST
# -------------------------------------------------------------------------

figs_model_lock = {
    "notebook": 7,
    "test_set_opened": False,

    "selection_protocol": {
        "complexity_selected_using": (
            "4-fold expanding date-blocked "
            "temporal CV on 240 model-selection matches"
        ),
        "validation_role": (
            "forward temporal confirmation only; "
            "not used to change locked candidate"
        ),
        "test_matches": 79,
    },

    "classification": {
        "feature_set": (
            SELECTED_FIGS_CLASS_FEATURE_SET
        ),
        "n_features": len(
            FEATURE_SETS[
                SELECTED_FIGS_CLASS_FEATURE_SET
            ]
        ),
        "max_rules": (
            SELECTED_FIGS_CLASS_MAX_RULES
        ),
        "cv_mean_rps": float(
            selected_classification_cv_row[
                "mean_rps"
            ]
        ),
        "validation_metrics": {
            key: float(value)
            for key, value
            in locked_figs_classifier_metrics.items()
        },
        "selected_feature_concept": (
            "leakage"
        ),
        "matched_cart_rps": float(
            cart_class_metrics[
                "rps"
            ]
        ),
    },

    "regression": {
        "feature_set": (
            SELECTED_FIGS_REG_FEATURE_SET
        ),
        "n_features": len(
            FEATURE_SETS[
                SELECTED_FIGS_REG_FEATURE_SET
            ]
        ),
        "max_rules": (
            SELECTED_FIGS_REG_MAX_RULES
        ),
        "cv_mean_mae": float(
            selected_regression_cv_row[
                "mean_mae"
            ]
        ),
        "validation_metrics": {
            key: float(value)
            for key, value
            in locked_figs_regressor_metrics.items()
        },
        "selected_feature_concept": (
            "completed_passes"
        ),
        "matched_cart_mae": float(
            cart_reg_metrics[
                "mae"
            ]
        ),
    },

    "environment": {
        "python": (
            sys.version.split()[0]
        ),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": (
            sklearn.__version__
        ),
        "imodels": version(
            "imodels"
        ),
    },
}


FIGS_MODEL_LOCK_PATH = (
    FIGS_RESULTS_DIR
    / "figs_model_lock_before_test.json"
)


with open(
    FIGS_MODEL_LOCK_PATH,
    "w",
    encoding="utf-8",
) as file_handle:

    json.dump(
        figs_model_lock,
        file_handle,
        indent=2,
    )


print(
    "FIGS development decisions LOCKED."
)

print(
    "Classification:",
    SELECTED_FIGS_CLASS_FEATURE_SET,
    "| max_rules =",
    SELECTED_FIGS_CLASS_MAX_RULES,
)

print(
    "Regression    :",
    SELECTED_FIGS_REG_FEATURE_SET,
    "| max_rules =",
    SELECTED_FIGS_REG_MAX_RULES,
)

print(
    "Held-out test : SEALED"
)

print(
    "\nLock file:",
    FIGS_MODEL_LOCK_PATH
)

FIGS development decisions LOCKED.
Classification: conventional_plus_all_network | max_rules = 2
Regression    : conventional_plus_possession_flow | max_rules = 2
Held-out test : SEALED

Lock file: ../data/processed/figs_results/figs_model_lock_before_test.json


Final held-out test opening.
All FIGS feature families, split budgets, candidate models, matched-CART comparisons and temporal-stability analyses were completed and locked before executing this cell. No decision after this point may be changed in response to test performance.

In [64]:
# -------------------------------------------------------------------------
# OPEN FINAL HELD-OUT TEST — ONCE ONLY
# -------------------------------------------------------------------------

assert FIGS_MODEL_LOCK_PATH.exists()

with open(
    FIGS_MODEL_LOCK_PATH,
    "r",
    encoding="utf-8",
) as file_handle:

    locked_config = json.load(
        file_handle
    )


assert (
    locked_config[
        "test_set_opened"
    ]
    is False
)


# -------------------------------------------------------------------------
# RECONSTRUCT DEVELOPMENT AND TEST FROM CANONICAL DATA
# -------------------------------------------------------------------------

development_df = (
    modeling_df.loc[
        modeling_df[
            "clean_date"
        ] < TEST_START_DATE
    ]
    .copy()
    .reset_index(drop=True)
)


final_test_df = (
    modeling_df.loc[
        modeling_df[
            "clean_date"
        ] >= TEST_START_DATE
    ]
    .copy()
    .reset_index(drop=True)
)


assert len(
    development_df
) == 301

assert len(
    final_test_df
) == 79


assert (
    development_df[
        "clean_date"
    ].max()
    <
    final_test_df[
        "clean_date"
    ].min()
)


print(
    "FINAL TEST OPENED."
)

print(
    f"Development matches : {len(development_df)}"
)

print(
    f"Final test matches  : {len(final_test_df)}"
)

print(
    "Development end    :",
    development_df[
        "clean_date"
    ].max().date(),
)

print(
    "Test start         :",
    final_test_df[
        "clean_date"
    ].min().date(),
)

FINAL TEST OPENED.
Development matches : 301
Final test matches  : 79
Development end    : 2016-04-01
Test start         : 2016-04-02


In [65]:
# -------------------------------------------------------------------------
# FINAL DEVELOPMENT / TEST TARGETS
# -------------------------------------------------------------------------

y_class_development = (
    development_df[
        "target_outcome"
    ]
    .map(
        CLASS_TO_INT
    )
    .astype(
        "int8"
    )
    .to_numpy()
)


y_class_test = (
    final_test_df[
        "target_outcome"
    ]
    .map(
        CLASS_TO_INT
    )
    .astype(
        "int8"
    )
    .to_numpy()
)


y_margin_development = (
    development_df[
        "target_margin"
    ]
    .to_numpy(
        dtype=float
    )
)


y_margin_test = (
    final_test_df[
        "target_margin"
    ]
    .to_numpy(
        dtype=float
    )
)


print(
    "Final held-out targets materialized."
)

print(
    "Classification test shape:",
    y_class_test.shape,
)

print(
    "Regression test shape    :",
    y_margin_test.shape,
)

Final held-out targets materialized.
Classification test shape: (79,)
Regression test shape    : (79,)


In [66]:
# -------------------------------------------------------------------------
# FINAL FIGS CLASSIFIER REFIT
# -------------------------------------------------------------------------

final_class_feature_columns = (
    FEATURE_SETS[
        SELECTED_FIGS_CLASS_FEATURE_SET
    ]
)


final_class_imputer = SimpleImputer(
    strategy="median",
    keep_empty_features=True,
)


X_class_development = (
    final_class_imputer.fit_transform(
        development_df[
            final_class_feature_columns
        ]
    )
)


X_class_test = (
    final_class_imputer.transform(
        final_test_df[
            final_class_feature_columns
        ]
    )
)


final_figs_classifier = FIGSClassifier(
    max_rules=(
        SELECTED_FIGS_CLASS_MAX_RULES
    ),
    random_state=RANDOM_STATE,
)


final_figs_classifier.fit(
    X_class_development,
    y_class_development,
)


final_class_probabilities = (
    align_class_probabilities(
        final_figs_classifier.predict_proba(
            X_class_test
        ),
        final_figs_classifier.classes_,
    )
)


final_classification_metrics = (
    evaluate_classification_probabilities(
        y_class_test,
        final_class_probabilities,
    )
)


print(
    "FINAL FIGS CLASSIFICATION TEST"
)

for metric_name, metric_value in (
    final_classification_metrics.items()
):

    print(
        f"{metric_name:20s}: "
        f"{metric_value:.4f}"
    )

FINAL FIGS CLASSIFICATION TEST
rps                 : 0.2177
log_loss            : 1.0190
brier               : 0.6115
ece                 : 0.1414
accuracy            : 0.4684
balanced_accuracy   : 0.5165
macro_f1            : 0.4800


In [67]:
# -------------------------------------------------------------------------
# FINAL FIGS REGRESSOR REFIT
# -------------------------------------------------------------------------

final_reg_feature_columns = (
    FEATURE_SETS[
        SELECTED_FIGS_REG_FEATURE_SET
    ]
)


final_reg_imputer = SimpleImputer(
    strategy="median",
    keep_empty_features=True,
)


X_reg_development = (
    final_reg_imputer.fit_transform(
        development_df[
            final_reg_feature_columns
        ]
    )
)


X_reg_test = (
    final_reg_imputer.transform(
        final_test_df[
            final_reg_feature_columns
        ]
    )
)


final_figs_regressor = FIGSRegressor(
    max_rules=(
        SELECTED_FIGS_REG_MAX_RULES
    ),
    random_state=RANDOM_STATE,
)


final_figs_regressor.fit(
    X_reg_development,
    y_margin_development,
)


final_margin_predictions = (
    clip_margin_predictions(
        final_figs_regressor.predict(
            X_reg_test
        )
    )
)


final_regression_metrics = (
    evaluate_regression_predictions(
        y_margin_test,
        final_margin_predictions,
    )
)


print(
    "FINAL FIGS REGRESSION TEST"
)

for metric_name, metric_value in (
    final_regression_metrics.items()
):

    print(
        f"{metric_name:15s}: "
        f"{metric_value:.4f}"
    )

FINAL FIGS REGRESSION TEST
mae            : 1.3527
rmse           : 1.7479
correlation    : 0.4275


In [68]:
# -------------------------------------------------------------------------
# SAVE FINAL FIGS TEST METRICS
# -------------------------------------------------------------------------

final_figs_test_results = {
    "test_set_opened": True,
    "test_matches": int(
        len(final_test_df)
    ),

    "classification": {
        "feature_set": (
            SELECTED_FIGS_CLASS_FEATURE_SET
        ),
        "n_features": len(
            final_class_feature_columns
        ),
        "max_rules": (
            SELECTED_FIGS_CLASS_MAX_RULES
        ),
        "metrics": {
            key: float(value)
            for key, value
            in final_classification_metrics.items()
        },
    },

    "regression": {
        "feature_set": (
            SELECTED_FIGS_REG_FEATURE_SET
        ),
        "n_features": len(
            final_reg_feature_columns
        ),
        "max_rules": (
            SELECTED_FIGS_REG_MAX_RULES
        ),
        "metrics": {
            key: float(value)
            for key, value
            in final_regression_metrics.items()
        },
    },
}


FINAL_FIGS_TEST_RESULTS_PATH = (
    FIGS_RESULTS_DIR
    / "figs_final_test_results.json"
)


with open(
    FINAL_FIGS_TEST_RESULTS_PATH,
    "w",
    encoding="utf-8",
) as file_handle:

    json.dump(
        final_figs_test_results,
        file_handle,
        indent=2,
    )


print(
    "Final FIGS test results saved."
)

print(
    FINAL_FIGS_TEST_RESULTS_PATH
)

Final FIGS test results saved.
../data/processed/figs_results/figs_final_test_results.json


In [69]:
# -------------------------------------------------------------------------
# FINAL FIGS TEST SUMMARY
# -------------------------------------------------------------------------

final_figs_test_summary_df = pd.DataFrame(
    [
        {
            "task": "classification",
            "feature_set": (
                SELECTED_FIGS_CLASS_FEATURE_SET
            ),
            "n_features": len(
                final_class_feature_columns
            ),
            "max_rules": (
                SELECTED_FIGS_CLASS_MAX_RULES
            ),
            "primary_metric": "RPS",
            "primary_value": (
                final_classification_metrics[
                    "rps"
                ]
            ),
            "secondary_1": (
                final_classification_metrics[
                    "log_loss"
                ]
            ),
            "secondary_2": (
                final_classification_metrics[
                    "brier"
                ]
            ),
        },
        {
            "task": "regression",
            "feature_set": (
                SELECTED_FIGS_REG_FEATURE_SET
            ),
            "n_features": len(
                final_reg_feature_columns
            ),
            "max_rules": (
                SELECTED_FIGS_REG_MAX_RULES
            ),
            "primary_metric": "MAE",
            "primary_value": (
                final_regression_metrics[
                    "mae"
                ]
            ),
            "secondary_1": (
                final_regression_metrics[
                    "rmse"
                ]
            ),
            "secondary_2": (
                final_regression_metrics[
                    "correlation"
                ]
            ),
        },
    ]
)


display(
    final_figs_test_summary_df
)

,task,feature_set,n_features,max_rules,primary_metric,primary_value,secondary_1,secondary_2
0,classification,conventional_plus_all_network,186,2,RPS,0.2177,1.0190,0.6115
1,regression,conventional_plus_possession_flow,42,2,MAE,1.3527,1.7479,0.4275


In [70]:
# -------------------------------------------------------------------------
# FINAL REFITTED FIGS CLASSIFIER STRUCTURE
# -------------------------------------------------------------------------

print(
    final_figs_classifier
)


final_class_feature_importance_df = (
    pd.DataFrame(
        {
            "feature": (
                final_class_feature_columns
            ),
            "importance": (
                final_figs_classifier
                .feature_importances_
            ),
        }
    )
    .loc[
        lambda df:
        df[
            "importance"
        ] > 0
    ]
    .sort_values(
        "importance",
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "\nFinal classification selected features:"
)

display(
    final_class_feature_importance_df
)

> ------------------------------
> FIGS-Fast Interpretable Greedy-Tree Sums:
> 	Predictions are made by summing the "Val" reached by traversing each tree.
> 	For classifiers, a softmax function is then applied to the sum.
> ------------------------------
X177 <= -0.031 (Tree #0 root)
	Val: 0.083 0.119 0.798 (leaf)
	X119 <= 0.886 (split)
		Val: 0.533 0.174 0.293 (leaf)
		Val: 0.216 0.392 0.392 (leaf)


Final classification selected features:


,feature,importance
0,diff_mean_zone_leakage_expanding_mean,0.6880
1,away_maximum_zone_leakage_expanding_mean,0.3120


In [71]:
# -------------------------------------------------------------------------
# FINAL REFITTED FIGS REGRESSOR STRUCTURE
# -------------------------------------------------------------------------

print(
    final_figs_regressor
)


final_reg_feature_importance_df = (
    pd.DataFrame(
        {
            "feature": (
                final_reg_feature_columns
            ),
            "importance": (
                final_figs_regressor
                .feature_importances_
            ),
        }
    )
    .loc[
        lambda df:
        df[
            "importance"
        ] > 0
    ]
    .sort_values(
        "importance",
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "\nFinal regression selected features:"
)

display(
    final_reg_feature_importance_df
)

> ------------------------------
> FIGS-Fast Interpretable Greedy-Tree Sums:
> 	Predictions are made by summing the "Val" reached by traversing each tree.
> 	For classifiers, a softmax function is then applied to the sum.
> ------------------------------
X36 <= 124.050 (Tree #0 root)
	X36 <= -249.000 (split)
		Val: -2.6 (leaf)
		Val: 0.289 (leaf)
	Val: 2.579 (leaf)


Final regression selected features:


,feature,importance
0,diff_completed_passes_roll10_mean,1.0000


In [72]:
# -------------------------------------------------------------------------
# FINAL TEST BOOKMAKER BENCHMARK
# -------------------------------------------------------------------------

market_test_probabilities = (
    final_test_df[
        [
            "market_prob_A",
            "market_prob_D",
            "market_prob_H",
        ]
    ]
    .to_numpy(
        dtype=float
    )
)


assert market_test_probabilities.shape == (
    len(
        final_test_df
    ),
    3,
)

assert np.isfinite(
    market_test_probabilities
).all()

assert np.allclose(
    market_test_probabilities.sum(
        axis=1
    ),
    1.0,
    atol=1e-6,
)


final_market_metrics = (
    evaluate_classification_probabilities(
        y_class_test,
        market_test_probabilities,
    )
)


print(
    "FINAL BOOKMAKER CLASSIFICATION TEST"
)

for metric_name, metric_value in (
    final_market_metrics.items()
):

    print(
        f"{metric_name:20s}: "
        f"{metric_value:.4f}"
    )

FINAL BOOKMAKER CLASSIFICATION TEST
rps                 : 0.1895
log_loss            : 0.9230
brier               : 0.5381
ece                 : 0.0666
accuracy            : 0.5823
balanced_accuracy   : 0.4652
macro_f1            : 0.4173


In [73]:
# -------------------------------------------------------------------------
# FINAL TEST — FIGS VS BOOKMAKER
# -------------------------------------------------------------------------

final_figs_vs_market_df = pd.DataFrame(
    [
        {
            "model": "Bookmaker",
            **final_market_metrics,
        },
        {
            "model": "FIGS",
            **final_classification_metrics,
        },
    ]
).sort_values(
    "rps",
    ascending=True,
).reset_index(
    drop=True
)


display(
    final_figs_vs_market_df
)

,model,rps,log_loss,brier,ece,accuracy,balanced_accuracy,macro_f1
0,Bookmaker,0.1895,0.9230,0.5381,0.0666,0.5823,0.4652,0.4173
1,FIGS,0.2177,1.0190,0.6115,0.1414,0.4684,0.5165,0.4800


In [74]:
# -------------------------------------------------------------------------
# INSPECT FROZEN NOTEBOOK 06 MODEL SPECIFICATIONS
# -------------------------------------------------------------------------

print(
    "Top-level keys:"
)

print(
    baseline_selection.keys()
)


print(
    "\nClassification selection:"
)

print(
    json.dumps(
        baseline_selection[
            "classification"
        ],
        indent=2,
    )
)


print(
    "\nRegression selection:"
)

print(
    json.dumps(
        baseline_selection[
            "regression"
        ],
        indent=2,
    )
)

Top-level keys:
dict_keys(['classification', 'regression', 'development_protocol', 'features', 'test_set_opened'])

Classification selection:
{
  "selected_model": "RandomForest",
  "selection_metric": "validation_rps",
  "best_params": {
    "model__max_depth": 4,
    "model__max_features": 0.5,
    "model__min_samples_leaf": 5
  },
  "validation_rps": 0.21055091449889465,
  "validation_log_loss": 1.0266969316843648,
  "validation_brier": 0.6114572152531469,
  "validation_ece": 0.09759267165993991
}

Regression selection:
{
  "selected_model": "RandomForest",
  "selection_metric": "validation_mae",
  "best_params": {
    "model__max_depth": null,
    "model__max_features": "sqrt",
    "model__min_samples_leaf": 5
  },
  "validation_mae": 1.358271768989331,
  "validation_rmse": 1.687856751280965,
  "validation_correlation": 0.5354881961386267
}


In [75]:
# -------------------------------------------------------------------------
# FINAL BASELINE IMPORTS
# -------------------------------------------------------------------------

from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor,
)

print(
    "Final Random Forest baseline imports passed."
)

Final Random Forest baseline imports passed.


In [76]:
# -------------------------------------------------------------------------
# FROZEN NOTEBOOK 06 RANDOM FOREST SPECIFICATIONS
# -------------------------------------------------------------------------

assert (
    baseline_selection[
        "classification"
    ][
        "selected_model"
    ]
    == "RandomForest"
)

assert (
    baseline_selection[
        "regression"
    ][
        "selected_model"
    ]
    == "RandomForest"
)


RF_CLASS_PARAMS = {
    "n_estimators": 300,
    "max_depth": (
        baseline_selection[
            "classification"
        ][
            "best_params"
        ][
            "model__max_depth"
        ]
    ),
    "max_features": (
        baseline_selection[
            "classification"
        ][
            "best_params"
        ][
            "model__max_features"
        ]
    ),
    "min_samples_leaf": (
        baseline_selection[
            "classification"
        ][
            "best_params"
        ][
            "model__min_samples_leaf"
        ]
    ),
    "random_state": RANDOM_STATE,
    "n_jobs": 1,
}


RF_REG_PARAMS = {
    "n_estimators": 300,
    "max_depth": (
        baseline_selection[
            "regression"
        ][
            "best_params"
        ][
            "model__max_depth"
        ]
    ),
    "max_features": (
        baseline_selection[
            "regression"
        ][
            "best_params"
        ][
            "model__max_features"
        ]
    ),
    "min_samples_leaf": (
        baseline_selection[
            "regression"
        ][
            "best_params"
        ][
            "model__min_samples_leaf"
        ]
    ),
    "random_state": RANDOM_STATE,
    "n_jobs": 1,
}


print(
    "Frozen RF classifier:"
)

print(
    RF_CLASS_PARAMS
)


print(
    "\nFrozen RF regressor:"
)

print(
    RF_REG_PARAMS
)

Frozen RF classifier:
{'n_estimators': 300, 'max_depth': 4, 'max_features': 0.5, 'min_samples_leaf': 5, 'random_state': 42, 'n_jobs': 1}

Frozen RF regressor:
{'n_estimators': 300, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'random_state': 42, 'n_jobs': 1}


In [77]:
# -------------------------------------------------------------------------
# FINAL NOTEBOOK 06 RANDOM FOREST CLASSIFIER REFIT
# -------------------------------------------------------------------------

baseline_feature_columns = (
    FULL_FEATURE_COLUMNS
)


final_rf_class_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                keep_empty_features=True,
            ),
        ),
        (
            "model",
            RandomForestClassifier(
                **RF_CLASS_PARAMS
            ),
        ),
    ]
)


X_rf_class_development = (
    development_df[
        baseline_feature_columns
    ]
)

X_rf_class_test = (
    final_test_df[
        baseline_feature_columns
    ]
)


final_rf_class_pipeline.fit(
    X_rf_class_development,
    y_class_development,
)


final_rf_class_raw_probabilities = (
    final_rf_class_pipeline.predict_proba(
        X_rf_class_test
    )
)


final_rf_class_probabilities = (
    align_class_probabilities(
        final_rf_class_raw_probabilities,
        final_rf_class_pipeline[
            "model"
        ].classes_,
    )
)


final_rf_classification_metrics = (
    evaluate_classification_probabilities(
        y_class_test,
        final_rf_class_probabilities,
    )
)


print(
    "FINAL RANDOM FOREST CLASSIFICATION TEST"
)

for metric_name, metric_value in (
    final_rf_classification_metrics.items()
):

    print(
        f"{metric_name:20s}: "
        f"{metric_value:.4f}"
    )

FINAL RANDOM FOREST CLASSIFICATION TEST
rps                 : 0.2143
log_loss            : 1.0152
brier               : 0.6058
ece                 : 0.1676
accuracy            : 0.5443
balanced_accuracy   : 0.4446
macro_f1            : 0.4289


In [78]:
# -------------------------------------------------------------------------
# FINAL NOTEBOOK 06 RANDOM FOREST REGRESSOR REFIT
# -------------------------------------------------------------------------

final_rf_reg_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                keep_empty_features=True,
            ),
        ),
        (
            "model",
            RandomForestRegressor(
                **RF_REG_PARAMS
            ),
        ),
    ]
)


X_rf_reg_development = (
    development_df[
        baseline_feature_columns
    ]
)

X_rf_reg_test = (
    final_test_df[
        baseline_feature_columns
    ]
)


final_rf_reg_pipeline.fit(
    X_rf_reg_development,
    y_margin_development,
)


final_rf_margin_predictions = (
    clip_margin_predictions(
        final_rf_reg_pipeline.predict(
            X_rf_reg_test
        )
    )
)


final_rf_regression_metrics = (
    evaluate_regression_predictions(
        y_margin_test,
        final_rf_margin_predictions,
    )
)


print(
    "FINAL RANDOM FOREST REGRESSION TEST"
)

for metric_name, metric_value in (
    final_rf_regression_metrics.items()
):

    print(
        f"{metric_name:15s}: "
        f"{metric_value:.4f}"
    )

FINAL RANDOM FOREST REGRESSION TEST
mae            : 1.3686
rmse           : 1.6995
correlation    : 0.4756


In [79]:
# -------------------------------------------------------------------------
# FINAL HELD-OUT CLASSIFICATION COMPARISON
# -------------------------------------------------------------------------

final_classification_comparison_df = (
    pd.DataFrame(
        [
            {
                "model": "Bookmaker",
                "role": "market_baseline",
                **final_market_metrics,
            },
            {
                "model": "RandomForest",
                "role": "notebook06_selected_ml",
                **final_rf_classification_metrics,
            },
            {
                "model": "FIGS",
                "role": "p2_interpretable",
                **final_classification_metrics,
            },
        ]
    )
    .sort_values(
        "rps",
        ascending=True,
    )
    .reset_index(drop=True)
)


display(
    final_classification_comparison_df
)

,model,role,rps,log_loss,brier,ece,accuracy,balanced_accuracy,macro_f1
0,Bookmaker,market_baseline,0.1895,0.9230,0.5381,0.0666,0.5823,0.4652,0.4173
1,RandomForest,notebook06_selected_ml,0.2143,1.0152,0.6058,0.1676,0.5443,0.4446,0.4289
2,FIGS,p2_interpretable,0.2177,1.0190,0.6115,0.1414,0.4684,0.5165,0.4800


In [80]:
# -------------------------------------------------------------------------
# FINAL HELD-OUT REGRESSION COMPARISON
# -------------------------------------------------------------------------

final_regression_comparison_df = (
    pd.DataFrame(
        [
            {
                "model": "RandomForest",
                "role": "notebook06_selected_ml",
                **final_rf_regression_metrics,
            },
            {
                "model": "FIGS",
                "role": "p2_interpretable",
                **final_regression_metrics,
            },
        ]
    )
    .sort_values(
        "mae",
        ascending=True,
    )
    .reset_index(drop=True)
)


display(
    final_regression_comparison_df
)

,model,role,mae,rmse,correlation
0,FIGS,p2_interpretable,1.3527,1.7479,0.4275
1,RandomForest,notebook06_selected_ml,1.3686,1.6995,0.4756


In [81]:
# -------------------------------------------------------------------------
# FINAL TEST PERFORMANCE DIFFERENCES
# -------------------------------------------------------------------------

classification_delta_vs_rf = (
    final_classification_metrics[
        "rps"
    ]
    - final_rf_classification_metrics[
        "rps"
    ]
)

classification_delta_vs_market = (
    final_classification_metrics[
        "rps"
    ]
    - final_market_metrics[
        "rps"
    ]
)

regression_delta_vs_rf = (
    final_regression_metrics[
        "mae"
    ]
    - final_rf_regression_metrics[
        "mae"
    ]
)


print(
    "Classification:"
)

print(
    "FIGS RPS - RF RPS     :",
    f"{classification_delta_vs_rf:+.4f}",
)

print(
    "FIGS RPS - Market RPS :",
    f"{classification_delta_vs_market:+.4f}",
)


print(
    "\nRegression:"
)

print(
    "FIGS MAE - RF MAE     :",
    f"{regression_delta_vs_rf:+.4f}",
)

Classification:
FIGS RPS - RF RPS     : +0.0034
FIGS RPS - Market RPS : +0.0283

Regression:
FIGS MAE - RF MAE     : -0.0160


In [82]:
# -------------------------------------------------------------------------
# SAVE FINAL HELD-OUT COMPARISONS
# -------------------------------------------------------------------------

FINAL_CLASSIFICATION_COMPARISON_PATH = (
    FIGS_RESULTS_DIR
    / "final_test_classification_comparison.csv"
)

FINAL_REGRESSION_COMPARISON_PATH = (
    FIGS_RESULTS_DIR
    / "final_test_regression_comparison.csv"
)


final_classification_comparison_df.to_csv(
    FINAL_CLASSIFICATION_COMPARISON_PATH,
    index=False,
)

final_regression_comparison_df.to_csv(
    FINAL_REGRESSION_COMPARISON_PATH,
    index=False,
)


print(
    "Final held-out comparison tables saved."
)

Final held-out comparison tables saved.


In [83]:
# -------------------------------------------------------------------------
# SAVE FINAL HELD-OUT PREDICTIONS
# -------------------------------------------------------------------------

final_test_predictions_df = (
    final_test_df[
        [
            "match_id",
            "clean_date",
            "home_team",
            "away_team",
            "target_outcome",
            "target_margin",
        ]
    ]
    .copy()
)


# -------------------------------------------------------------------------
# CLASSIFICATION
# -------------------------------------------------------------------------

final_test_predictions_df[
    "figs_prob_A"
] = final_class_probabilities[:, 0]

final_test_predictions_df[
    "figs_prob_D"
] = final_class_probabilities[:, 1]

final_test_predictions_df[
    "figs_prob_H"
] = final_class_probabilities[:, 2]


final_test_predictions_df[
    "rf_prob_A"
] = final_rf_class_probabilities[:, 0]

final_test_predictions_df[
    "rf_prob_D"
] = final_rf_class_probabilities[:, 1]

final_test_predictions_df[
    "rf_prob_H"
] = final_rf_class_probabilities[:, 2]


final_test_predictions_df[
    "market_prob_A"
] = market_test_probabilities[:, 0]

final_test_predictions_df[
    "market_prob_D"
] = market_test_probabilities[:, 1]

final_test_predictions_df[
    "market_prob_H"
] = market_test_probabilities[:, 2]


# -------------------------------------------------------------------------
# REGRESSION
# -------------------------------------------------------------------------

final_test_predictions_df[
    "figs_margin_prediction"
] = final_margin_predictions

final_test_predictions_df[
    "rf_margin_prediction"
] = final_rf_margin_predictions


FINAL_TEST_PREDICTIONS_PATH = (
    FIGS_RESULTS_DIR
    / "final_test_match_predictions.csv"
)


final_test_predictions_df.to_csv(
    FINAL_TEST_PREDICTIONS_PATH,
    index=False,
)


print(
    "Final held-out predictions saved."
)

print(
    FINAL_TEST_PREDICTIONS_PATH
)

display(
    final_test_predictions_df.head()
)

Final held-out predictions saved.
../data/processed/figs_results/final_test_match_predictions.csv


,match_id,clean_date,home_team,away_team,target_outcome,target_margin,figs_prob_A,figs_prob_D,figs_prob_H,rf_prob_A,rf_prob_D,rf_prob_H,market_prob_A,market_prob_D,market_prob_H,figs_margin_prediction,rf_margin_prediction
0,267533,2016-04-02,Barcelona,Real Madrid,A,-1,0.4023,0.2810,0.3167,0.3278,0.1958,0.4764,0.1747,0.2135,0.6119,0.2885,0.9240
1,3825822,2016-04-02,Las Palmas,Valencia,H,1,0.2452,0.2541,0.5008,0.1068,0.1454,0.7478,0.3071,0.2929,0.4000,0.2885,1.2344
2,3825872,2016-04-02,Atlético Madrid,Real Betis,H,4,0.2452,0.2541,0.5008,0.1047,0.1150,0.7803,0.0957,0.2210,0.6834,0.2885,1.2111
3,3825898,2016-04-02,Celta Vigo,RC Deportivo La Coruña,D,0,0.2452,0.2541,0.5008,0.1430,0.2089,0.6482,0.2014,0.2518,0.5468,0.2885,1.2242
4,3825823,2016-04-03,Sevilla,Real Sociedad,A,-1,0.4023,0.2810,0.3167,0.1943,0.2152,0.5905,0.1733,0.2383,0.5884,0.2885,0.7993


In [84]:
# -------------------------------------------------------------------------
# SAVE FINAL FIGS MODELS
# -------------------------------------------------------------------------

import joblib


FINAL_FIGS_CLASSIFIER_PATH = (
    FIGS_MODEL_DIR
    / "final_figs_classifier.joblib"
)

FINAL_FIGS_REGRESSOR_PATH = (
    FIGS_MODEL_DIR
    / "final_figs_regressor.joblib"
)

FINAL_FIGS_CLASS_IMPUTER_PATH = (
    FIGS_MODEL_DIR
    / "final_figs_classifier_imputer.joblib"
)

FINAL_FIGS_REG_IMPUTER_PATH = (
    FIGS_MODEL_DIR
    / "final_figs_regressor_imputer.joblib"
)


joblib.dump(
    final_figs_classifier,
    FINAL_FIGS_CLASSIFIER_PATH,
)

joblib.dump(
    final_figs_regressor,
    FINAL_FIGS_REGRESSOR_PATH,
)

joblib.dump(
    final_class_imputer,
    FINAL_FIGS_CLASS_IMPUTER_PATH,
)

joblib.dump(
    final_reg_imputer,
    FINAL_FIGS_REG_IMPUTER_PATH,
)


print(
    "Final FIGS models saved."
)

Final FIGS models saved.


In [85]:
# -------------------------------------------------------------------------
# NOTEBOOK 07 FINAL AUDIT
# -------------------------------------------------------------------------

notebook07_final_audit = {
    "notebook": 7,
    "status": "complete",
    "test_set_opened": True,

    "development_matches": int(
        len(development_df)
    ),

    "test_matches": int(
        len(final_test_df)
    ),

    "classification": {
        "selected_feature_set": (
            SELECTED_FIGS_CLASS_FEATURE_SET
        ),
        "max_rules": (
            SELECTED_FIGS_CLASS_MAX_RULES
        ),
        "dominant_network_concept": "leakage",
        "figs_test_metrics": {
            key: float(value)
            for key, value
            in final_classification_metrics.items()
        },
        "random_forest_test_metrics": {
            key: float(value)
            for key, value
            in final_rf_classification_metrics.items()
        },
        "market_test_metrics": {
            key: float(value)
            for key, value
            in final_market_metrics.items()
        },
    },

    "regression": {
        "selected_feature_set": (
            SELECTED_FIGS_REG_FEATURE_SET
        ),
        "max_rules": (
            SELECTED_FIGS_REG_MAX_RULES
        ),
        "dominant_network_concept": (
            "completed_passes"
        ),
        "figs_test_metrics": {
            key: float(value)
            for key, value
            in final_regression_metrics.items()
        },
        "random_forest_test_metrics": {
            key: float(value)
            for key, value
            in final_rf_regression_metrics.items()
        },
    },

    "protocol": {
        "post_test_retuning": False,
        "feature_selection_after_test": False,
        "complexity_selection_after_test": False,
    },

    "environment": {
        "python": sys.version.split()[0],
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "imodels": version("imodels"),
    },
}


NOTEBOOK07_FINAL_AUDIT_PATH = (
    FIGS_RESULTS_DIR
    / "notebook07_final_audit.json"
)


with open(
    NOTEBOOK07_FINAL_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as file_handle:

    json.dump(
        notebook07_final_audit,
        file_handle,
        indent=2,
    )


print(
    "Notebook 07 final audit saved."
)

print(
    NOTEBOOK07_FINAL_AUDIT_PATH
)

Notebook 07 final audit saved.
../data/processed/figs_results/notebook07_final_audit.json
